# mCPT replication for SemEval-2027 Task 9, Subtask 1 (paragraph-level framing)

A PyTorch + Hugging Face reimplementation of **mCPT** (Ertl, Reiter-Haas, Innerebner & Lex,
*SemEval-2023 Task 3*, [paper](https://aclanthology.org/2023.semeval-1.130/),
[reference code](https://github.com/socialcomplab/semeval23-mcpt)), adapted from article-level
(SemEval-2023 Subtask 2) to paragraph-level multi-label framing detection (SemEval-2027 Task 9, Subtask 1).

**How to use**
1. `Runtime → Change runtime type → GPU`.
2. Leave `SMOKE_TEST = True` for a fast end-to-end check on synthetic data with a tiny random encoder
   (about 1–2 min, no data or downloads needed).
3. Upload `data-fourth-batch-russian-2026_11_1.zip` to `/content/` (or set `ZIP_PATH`), set `SMOKE_TEST = False`,
   run all. The release folders are discovered automatically; section 3 prints a leakage audit before training.
   `EN_ONLY = True` (default) trains on all labelled data but builds, evaluates and submits only the EN model.

Every place where this notebook had to choose something not stated in the paper or task description is
tagged `ASSUMPTION` in the code and listed in the README. Places where the paper and the reference code
disagree are tagged `REF-DIFF`.

| Section | What it does |
|---|---|
| 1 Setup & config | installs, seeds, all hyperparameters in one dataclass |
| 2 Labels | the 14 Card et al. frames, name normalisation |
| 3 Data | 2027 paragraph loader, 2023 article loader, synthetic data, article-grouped splits |
| 4 Model | MiniLM encoder + mean pooling + head |
| 5 Loss | HeroCon-style label-aware contrastive loss (log-space, stable) |
| 6 Sampler | contrast sampler ("CS" variant) |
| 7 Trainer | head-only stages, contrastive stage, checkpoints, logs |
| 8 Evaluation & export | threshold tuning, micro/macro F1, official submission format |
| 9 Unit tests | weights frozen/changed, finite loss, export format, split leakage |
| 10 Pipelines | Phase 1, Phase 2 few-shot, Phase 2 zero-shot |
| 11 Ablations | −PT, −L_CON, −E2E, +CS |
| 12 Predict & export | dev/test submission files |

## 1. Setup & config

In [ ]:
# Colab already ships torch / transformers / sklearn / pandas. Pin nothing exotic.
import importlib.util, subprocess, sys
for pkg in ["transformers", "sklearn", "pandas"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "scikit-learn" if pkg == "sklearn" else pkg], check=True)

In [ ]:
import copy, dataclasses, glob, json, math, os, random, re, time
from dataclasses import dataclass, field, asdict
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import ExponentialLR
from torch.utils.data import DataLoader, Dataset, RandomSampler, Sampler
from sklearn.metrics import f1_score
from sklearn.model_selection import GroupShuffleSplit
from transformers import AutoModel, AutoTokenizer

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| torch", torch.__version__)

device: cuda | torch 2.14.1+cu130


In [ ]:
# ---- Optional: Google Drive for data + persistent checkpoints --------------------------------
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
SMOKE_TEST = False   # True: synthetic data + tiny random encoder, 1-2 epochs per stage. Set False for real runs.
REPORT_LANG = "EN"  # this language's dev score is printed separately in every evaluation line
EN_ONLY = True      # True: build/evaluate/submit the EN model only (all labelled data is still used for training);
                    # skips BG/PT/RU few-shot models, zero-shot and ablations.
FINAL_REFIT_WITH_DEV = True  # EN_ONLY: after the honest dev evaluation, retrain the best EN setup on EN train + EN
                             # labelled dev and write an extra "_FULLFIT" submission (threshold from the honest run)


@dataclass
class Config:
    # ---------------- data ----------------
    # Root of the unzipped release: the folder holding 1_initial-training-data/, 2_training-.../, 3_dev-set-.../,
    # 4_training-russian-.../. Every folder with a raw-documents/ subfolder is discovered:
    #   subtask-1-annotations.txt                              -> labelled training paragraphs
    #   subtask-1-annotations-for-subtask-2b-participant.txt   -> labelled paragraphs of dev-release docs (our dev)
    #   subtask-1-participant.txt                              -> unlabelled paragraphs to predict (leaderboard)
    data_dir: str = "/content/data"
    languages: List[str] = field(default_factory=lambda: ["BG", "EN", "PT", "RU"])   # task (target) languages
    use_semeval23_for_pretraining: bool = True  # SEMEVAL23_* folders (DE FR IT PL RU EN ES GR KA): phase 1 only
    dev_source: str = "gold_2b"             # "gold_2b": labelled dev-release docs where available, else internal split
                                            # "internal": always an article-grouped split of the 2027 train data
    internal_dev_frac: float = 0.2          # for languages without gold dev labels (RU in the Nov release)
    # ---- leakage control ----
    dedup_aware_split: bool = True          # articles sharing an identical (normalised) unit text go to the same split
    dedup_min_chars: int = 30               # shorter texts ("Photo: Reuters") never link articles
    dedup_max_articles: int = 5             # a text in more than this many articles = boilerplate: not used to link
    drop_dev_overlap: bool = True           # remove dev units whose text (near-)duplicates any train unit, all languages
    near_dup_threshold: float = 0.95        # TF-IDF char 3-5gram cosine at/above this counts as a duplicate

    # ---------------- model ----------------
    model_name: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    max_length: Optional[int] = None        # ASSUMPTION: None -> tokenizer.model_max_length capped at 512 (reference: truncation=True)
    head_hidden: int = 256
    head_dropout: float = 0.5
    head_variant: str = "paper"             # "paper": Linear-ReLU-Dropout-Linear | "reference_code": 2 hidden layers (REF-DIFF)

    # ---------------- loss ----------------
    alpha: float = 0.01                     # L = L_BCE + alpha * L_CON (reference: beta=0.01)
    con_temperature: float = 1.0            # f = exp(cos / T); reference uses T=1
    con_include_self_pairs: bool = False    # REF-DIFF: reference includes i==j pairs; spec asks to skip classes with <2 positives
    con_class_reduction: str = "mean"       # paper: mean over classes | reference code: "sum"
    con_min_positives: int = 2
    con_eps: float = 1e-8

    # ---------------- optimisation ----------------
    batch_size: int = 26                    # paper: 26 (reference uses 200 for head-only stages on cached embeddings: REF-DIFF)
    head_batch_size: int = 26
    eval_batch_size: int = 128
    head_lr: float = 1e-3
    encoder_lr: float = 2e-5
    use_lr_decay: bool = True               # reference code: ExponentialLR per epoch; paper does not mention it
    head_gamma: float = 0.99
    encoder_gamma: float = 0.98
    sampler: str = "random"                 # "random" (reference final run) | "contrast" (paper's CS variant)
    cs_min_samples_per_class: int = 1      # paper: "at least one sample from every class"

    # ---------------- schedule ----------------
    p1_head_epochs: int = 10                # paper
    p1_con_epochs: int = 50                 # paper
    p2_head_epochs: int = 10                # spec (paper does not state phase-2 epochs)
    p2_con_epochs: int = 50                 # ASSUMPTION from reference predictions notebook (N_FINETUNE_EPOCHS=50)
    p2_post_epochs: int = 50                # ASSUMPTION from reference predictions notebook (N_POST_FINETUNE_EPOCHS=50)
    zs_post_epochs: int = 50                # ASSUMPTION: same as p2_post_epochs
    restore_encoder_optimizer_in_p2: bool = True  # reference loads optimiser+scheduler state from the phase-1 checkpoint
    select_val_frac: float = 0.15           # EN push: article-grouped slice of TARGET train used only for epoch selection

    # ---------------- evaluation ----------------
    threshold_grid: Tuple[float, ...] = tuple(np.round(np.arange(0.05, 0.951, 0.05), 2))
    force_at_least_one: bool = False        # REF-DIFF: reference predicts argmax if nothing >= 0.5; wrong for empty paragraphs
    official_scorer_cmd: Optional[str] = None  # e.g. "python /content/scorer.py -g {gold} -p {pred}" (ASSUMPTION: CLI unknown)
    empty_label_trailing_tab: bool = True   # as in the release: no framing = "file\tstart\tend\t"
    eval_every: int = 5                     # dev metrics every N epochs during training (0 = only at stage end)

    # ---------------- bookkeeping ----------------
    seed: int = 42
    out_dir: str = "/content/runs"
    save_encoder_in_checkpoints: bool = True  # ~470 MB per checkpoint for MiniLM-L12
    num_workers: int = 2


CFG = Config()
if USE_DRIVE:
    CFG.out_dir = "/content/drive/MyDrive/mcpt_runs"

if SMOKE_TEST:
    CFG.data_dir = "/content/smoke_data" if os.path.isdir("/content") else os.path.abspath("smoke_data")
    CFG.out_dir = "/content/smoke_runs" if os.path.isdir("/content") else os.path.abspath("smoke_runs")
    CFG.p1_head_epochs, CFG.p1_con_epochs = 1, 2
    CFG.p2_head_epochs, CFG.p2_con_epochs, CFG.p2_post_epochs, CFG.zs_post_epochs = 1, 1, 1, 1
    CFG.eval_every = 1
    CFG.num_workers = 0
    CFG.save_encoder_in_checkpoints = True

os.makedirs(CFG.out_dir, exist_ok=True)
print(json.dumps(asdict(CFG), indent=1, default=str)[:1500])

{
 "data_dir": "/content/data",
 "languages": [
  "BG",
  "EN",
  "PT",
  "RU"
 ],
 "use_semeval23_for_pretraining": true,
 "dev_source": "gold_2b",
 "internal_dev_frac": 0.2,
 "dedup_aware_split": true,
 "dedup_min_chars": 30,
 "dedup_max_articles": 5,
 "drop_dev_overlap": true,
 "near_dup_threshold": 0.95,
 "model_name": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
 "max_length": null,
 "head_hidden": 256,
 "head_dropout": 0.5,
 "head_variant": "paper",
 "alpha": 0.01,
 "con_temperature": 1.0,
 "con_include_self_pairs": false,
 "con_class_reduction": "mean",
 "con_min_positives": 2,
 "con_eps": 1e-08,
 "batch_size": 26,
 "head_batch_size": 26,
 "eval_batch_size": 128,
 "head_lr": 0.001,
 "encoder_lr": 2e-05,
 "use_lr_decay": true,
 "head_gamma": 0.99,
 "encoder_gamma": 0.98,
 "sampler": "random",
 "cs_min_samples_per_class": 1,
 "p1_head_epochs": 10,
 "p1_con_epochs": 50,
 "p2_head_epochs": 10,
 "p2_con_epochs": 50,
 "p2_post_epochs": 50,
 "zs_post_epochs": 50,
 "re

In [ ]:
# ---- Locate / unzip the release (text only; subtask-2 images skipped) ------------------------------
# Finds the data automatically: (1) an already-unzipped release anywhere under SEARCH_ROOTS, else (2) any .zip there
# (including Colab's renamed "... (1).zip" uploads) that contains subtask-1-annotations.txt, extracted to EXTRACT_TO.
ZIP_PATH = "/content/data-fourth-batch-russian-2026_11_1.zip"   # optional explicit path (Files pane upload or Drive)
SEARCH_ROOTS = ["/content", "/content/drive/MyDrive"]
EXTRACT_TO = "/content/release"


def _release_dirs(root: str, max_depth: int = 7) -> List[str]:
    """Folders under root that contain a subtask-1-annotations.txt (bounded depth, skips Drive trash/system dirs)."""
    hits, root = [], os.path.abspath(root)
    for dirpath, dirs, files in os.walk(root):
        depth = dirpath[len(root):].count(os.sep)
        dirs[:] = [d for d in dirs if not d.startswith(".") and depth < max_depth
                   and d not in ("sample_data", "smoke_data", "smoke_runs", "runs", "drive")]
        if "subtask-1-annotations.txt" in files:
            hits.append(dirpath)
    return hits


def _common_root(paths: List[str]) -> str:
    """Release root = the folder above the batch folders (e.g. .../data)."""
    roots = []
    for p in paths:
        parts = p.split(os.sep)
        idx = next((i for i, x in enumerate(parts) if re.match(r"^\d+_", x)), None)   # 1_initial..., 2_training...
        roots.append(os.sep.join(parts[:idx]) if idx else os.path.dirname(os.path.dirname(p)))
    return os.path.commonpath(roots)


def locate_release(cfg: Config) -> str:
    # 1) already unzipped (configured dir first)
    for root in [cfg.data_dir, EXTRACT_TO] + SEARCH_ROOTS:
        if os.path.isdir(root):
            hits = _release_dirs(root, max_depth=4 if "drive" in root else 7)
            if hits:
                return _common_root(hits)
    # 2) find a zip that contains the ST1 files
    import zipfile
    cands = [ZIP_PATH] if os.path.isfile(ZIP_PATH) else []
    for root in SEARCH_ROOTS:
        cands += sorted(glob.glob(os.path.join(root, "*.zip")) + glob.glob(os.path.join(root, "*", "*.zip")))
    for zp in dict.fromkeys(cands):
        try:
            with zipfile.ZipFile(zp) as z:
                names = z.namelist()
                if not any(n.endswith("subtask-1-annotations.txt") for n in names):
                    continue
                members = [m for m in names if "/subtask2/" not in m
                           and not m.lower().endswith((".jpg", ".jpeg", ".png", ".webp", ".gif"))]
                print(f"extracting {len(members)} text files from {zp} -> {EXTRACT_TO}")
                z.extractall(EXTRACT_TO, members=members)
        except zipfile.BadZipFile:
            print(f"[warn] {zp} is not a complete zip (still uploading?)"); continue
        hits = _release_dirs(EXTRACT_TO)
        if hits:
            return _common_root(hits)
    raise FileNotFoundError(
        "Release not found. Upload data-fourth-batch-russian-2026_11_1.zip via the Files pane (wait until the upload "
        "finishes) or put it in Google Drive and set USE_DRIVE = True, then rerun this cell. Searched: "
        + ", ".join(SEARCH_ROOTS) + f"; zips seen: {cands or 'none'}")


if not SMOKE_TEST:
    CFG.data_dir = locate_release(CFG)
    print("data_dir =", CFG.data_dir)

data_dir = /content/release/data


In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(CFG.seed)

## 2. Labels
The 14 Card et al. (2015) frames. The 2027 label list uses spaces (and a typo, "resurces"), the gold example
and the 2023 data use underscores. Everything is normalised to the underscore form below, sorted
alphabetically as in the reference `INT2LABEL`.

In [ ]:
LABELS: List[str] = sorted([
    "Capacity_and_resources", "Crime_and_punishment", "Cultural_identity", "Economic",
    "External_regulation_and_reputation", "Fairness_and_equality", "Health_and_safety",
    "Legality_Constitutionality_and_jurisprudence", "Morality", "Policy_prescription_and_evaluation",
    "Political", "Public_opinion", "Quality_of_life", "Security_and_defense",
])
NUM_CLASSES = len(LABELS)
LABEL2ID = {l: i for i, l in enumerate(LABELS)}


def _key(s: str) -> str:
    return re.sub(r"[^a-z]", "", s.lower())


_ALIASES = {_key(l): l for l in LABELS}
_ALIASES[_key("Capacity and resurces")] = "Capacity_and_resources"   # typo in framing-labels.txt


def normalise_label(name: str) -> str:
    k = _key(name)
    if k not in _ALIASES:
        raise KeyError(f"Unknown framing label: {name!r}")
    return _ALIASES[k]


def labels_to_vec(names: Sequence[str]) -> np.ndarray:
    v = np.zeros(NUM_CLASSES, dtype=np.float32)
    for n in names:
        if n.strip() and n.strip() != "None":
            v[LABEL2ID[normalise_label(n)]] = 1.0
    return v


def vec_to_labels(v: Sequence[float]) -> List[str]:
    return [LABELS[i] for i, x in enumerate(v) if x >= 0.5]


assert NUM_CLASSES == 14

## 3. Data
Built for the actual release layout (`data/1_initial-training-data`, `2_training-second-batch-oct-revised`,
`3_dev-set-nov-release`, `4_training-russian-nov`). Facts checked on the November zip:
* Span files are `file \t start \t end \t label \t label ...`, CRLF line endings, and an **empty label field**
  (`file\tstart\tend\t`) for paragraphs with no framing.
* **Offsets index the raw file including `\r\n`**, so documents are read with `newline=""`. Reading with Python's
  default newline translation shifts every span after the first CRLF. The raw reading reproduces the
  `*-with-paragraphs.txt` text almost exactly (±1 char of whitespace normalisation in that file).
* The dev release ships gold ST1 labels for 20 extra docs per language
  (`subtask-1-annotations-for-subtask-2b-participant.txt`). Their doc names don't overlap the ST1
  leaderboard docs (`subtask-1-participant.txt`) or the train docs, so they make the labelled **dev** set.
  RU has none, so RU gets an article-grouped 20% split of its training data.
* SemEval-2023 folders (`SEMEVAL23_*`) have **no empty (NONE) paragraphs** and list only annotated paragraphs
  (README_ST1). They are used for multilingual phase-1 pre-training only, never for dev or few-shot.

One row per paragraph: `lang, origin, source, article_id, file, start, end, text, labels`.
`article_id = source/file` so identically named docs in different batches stay distinct.

In [ ]:
UNIT_COLS = ["lang", "origin", "source", "article_id", "file", "start", "end", "text", "labels"]


def _read_raw(path: str) -> str:
    with open(path, encoding="utf-8", newline="") as f:      # keep \r\n: offsets count them
        return f.read()


def _parse_span_file(path: str, with_labels: bool) -> List[Tuple[str, int, int, List[str]]]:
    rows = []
    with open(path, encoding="utf-8", newline="") as f:
        for ln, line in enumerate(f):
            line = line.rstrip("\n").rstrip("\r")
            if not line.strip():
                continue
            parts = line.split("\t")
            if len(parts) < 3:
                raise ValueError(f"{path}:{ln + 1}: expected >=3 tab-separated fields, got {parts}")
            labs = [p.strip() for p in parts[3:] if p.strip()] if with_labels else []
            rows.append((parts[0].strip(), int(parts[1]), int(parts[2]), labs))
    return rows


SPAN_FILES = {"subtask-1-annotations.txt": "train",
              "subtask-1-annotations-for-subtask-2b-participant.txt": "gold_dev",
              "subtask-1-participant.txt": "predict"}


def discover(cfg: Config) -> Dict[str, List[dict]]:
    out = {"train": [], "gold_dev": [], "predict": []}
    for dirpath, dirs, files in os.walk(cfg.data_dir):
        if "raw-documents" not in dirs:
            continue
        rel = os.path.relpath(dirpath, cfg.data_dir)
        src = dict(lang=os.path.basename(dirpath).upper(), dir=dirpath, rel=rel,
                   origin="2023" if "SEMEVAL23" in rel.upper() else "2027")
        for fn, kind in SPAN_FILES.items():
            if fn in files:
                out[kind].append(dict(src, path=os.path.join(dirpath, fn), kind=kind))
    for k in out:
        out[k].sort(key=lambda d: d["rel"])
    return out


def load_span_file(src: dict, labelled: bool) -> pd.DataFrame:
    rawdir = os.path.join(src["dir"], "raw-documents")
    cache: Dict[str, str] = {}
    recs = []
    for fname, s, e, labs in _parse_span_file(src["path"], with_labels=labelled):
        if fname not in cache:
            cache[fname] = _read_raw(os.path.join(rawdir, fname))
        recs.append(dict(lang=src["lang"], origin=src["origin"], source=src["rel"],
                         article_id=f"{src['rel']}/{fname}", file=fname, start=s, end=e,
                         text=cache[fname][s:e].strip(), labels=labels_to_vec(labs)))
    return pd.DataFrame(recs, columns=UNIT_COLS)


def _cat(dfs) -> pd.DataFrame:
    dfs = [d for d in dfs if len(d)]
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame(columns=UNIT_COLS)


def norm_text(t: str) -> str:
    return re.sub(r"\W+", " ", str(t).lower()).strip()


def duplicate_aware_groups(df: pd.DataFrame, cfg: "Config") -> np.ndarray:
    """Union-find over articles: two articles are linked if they share an identical normalised paragraph
    (republished / duplicated articles). Boilerplate (short, or in > dedup_max_articles articles) never links."""
    keys = df["article_id"].astype(str).values
    parent = {k: k for k in keys}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x

    by_text: Dict[str, set] = {}
    for k, t in zip(keys, df["text"].map(norm_text)):
        if len(t) >= cfg.dedup_min_chars:
            by_text.setdefault(t, set()).add(k)
    for arts in by_text.values():
        if 1 < len(arts) <= cfg.dedup_max_articles:
            a0 = next(iter(arts))
            for a in arts:
                parent[find(a)] = find(a0)
    return np.array([find(k) for k in keys])


def group_split(df: pd.DataFrame, frac: float, seed: int, cfg: Optional["Config"] = None) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Article-grouped split, per language. With cfg.dedup_aware_split, duplicated articles share a group."""
    tr, dv = [], []
    for lang, g in df.groupby("lang", sort=True):
        groups = duplicate_aware_groups(g, cfg) if (cfg is not None and cfg.dedup_aware_split) \
            else g["article_id"].astype(str).values
        if len(np.unique(groups)) < 2:
            tr.append(g); continue
        gss = GroupShuffleSplit(n_splits=1, test_size=frac, random_state=seed)
        i_tr, i_dv = next(gss.split(g, groups=groups))
        tr.append(g.iloc[i_tr]); dv.append(g.iloc[i_dv])
    return _cat(tr), _cat(dv)


def build_splits(cfg: Config, sources: Dict[str, List[dict]]) -> Dict[str, Dict[str, pd.DataFrame]]:
    """{lang: {"train": 2027 train (few-shot data), "dev": labelled dev, "extra": SemEval-2023 data (phase 1 only)}}"""
    tr27 = _cat([load_span_file(s, True) for s in sources["train"] if s["origin"] == "2027"])
    tr23 = _cat([load_span_file(s, True) for s in sources["train"] if s["origin"] == "2023"]) \
        if cfg.use_semeval23_for_pretraining else pd.DataFrame(columns=UNIT_COLS)
    gold = _cat([load_span_file(s, True) for s in sources["gold_dev"]])
    out = {}
    for lang in sorted(set(tr27["lang"]) | set(tr23["lang"])):
        t = tr27[tr27["lang"] == lang]
        g = gold[gold["lang"] == lang]
        e = tr23[tr23["lang"] == lang].reset_index(drop=True)
        if cfg.dev_source == "gold_2b" and len(g):
            dv, src = g.reset_index(drop=True), "gold ST1 labels of dev-release docs (2b file)"
        elif len(t):
            t, dv = group_split(t, cfg.internal_dev_frac, cfg.seed, cfg)
            src = f"article-grouped {cfg.internal_dev_frac:.0%} split of 2027 train"
        else:
            dv, src = pd.DataFrame(columns=UNIT_COLS), "none (pre-training language)"
        out[lang] = {"train": t.reset_index(drop=True), "dev": dv, "extra": e, "dev_source": src}
    return out


def pooled(splits, part: str, langs: Optional[Sequence[str]] = None, include_extra: bool = True) -> pd.DataFrame:
    langs = list(splits) if langs is None else [l for l in langs if l in splits]
    dfs = [splits[l][part] for l in langs]
    if part == "train" and include_extra:
        dfs += [splits[l]["extra"] for l in langs]
    return _cat(dfs)


def describe(df: pd.DataFrame, name: str) -> None:
    Y = np.stack(df["labels"].values) if len(df) else np.zeros((0, NUM_CLASSES))
    print(f"{name:>22}: {len(df):6d} units | {df['article_id'].nunique() if len(df) else 0:5d} articles | "
          f"empty {int((Y.sum(1) == 0).sum()):5d} | mean #labels {Y.sum(1).mean() if len(Y) else 0:.2f}")

### Leakage audit
Checks, for every dev unit, (i) shared article ids, (ii) identical normalised text and (iii) near-duplicate text
(TF-IDF char 3-5-gram cosine) against **all** training data that any phase sees (2027 train + SemEval-2023),
plus a majority-label baseline for scale. Overlapping dev units are then dropped (`drop_dev_overlap`).
Dev metrics also include a **cross-fitted** score (threshold tuned on one half of dev articles, scored on the
other), which is the number to trust; the plain tuned-on-dev score is optimistic.

In [ ]:
def train_dev_overlap(train_df: pd.DataFrame, dev_df: pd.DataFrame, threshold: float) -> Tuple[np.ndarray, np.ndarray]:
    """For every dev unit: (exact normalised-text match in train?, max TF-IDF char-ngram cosine to any train unit)."""
    from sklearn.feature_extraction.text import TfidfVectorizer
    tr_t, dv_t = train_df["text"].map(norm_text).tolist(), dev_df["text"].map(norm_text).tolist()
    trs = set(tr_t)
    exact = np.array([t in trs for t in dv_t], dtype=bool)
    if not len(dv_t) or not len(tr_t):
        return exact, np.zeros(len(dv_t))
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1, sublinear_tf=True).fit(tr_t + dv_t)
    A, B = vec.transform(dv_t), vec.transform(tr_t)          # rows are L2-normalised
    best = np.zeros(len(dv_t))
    for i in range(0, len(dv_t), 512):
        best[i:i + 512] = (A[i:i + 512] @ B.T).max(axis=1).toarray().ravel()
    return exact, best


def _official_micro(Y, Yh):   # micro-F1 with empty label set counted as "None" (as in section 8)
    add_none = lambda M: np.concatenate([M, (M.sum(1) == 0)[:, None]], 1).astype(int)
    Y, Yh = (np.asarray(Y) >= .5).astype(int), np.asarray(Yh).astype(int)
    return f1_score(add_none(Y), add_none(Yh), average="micro", zero_division=0)


def leakage_audit(splits, cfg: Config, extra_sets: Optional[Dict[str, pd.DataFrame]] = None) -> pd.DataFrame:
    """Any non-zero overlap column means dev scores are inflated."""
    all_train = pooled(splits, "train")
    sets = {f"{l}/dev": d["dev"] for l, d in splits.items() if len(d["dev"])}
    sets.update(extra_sets or {})
    rows = []
    for name, dv in sets.items():
        lang = name.split("/")[0]
        exact, sim = train_dev_overlap(all_train, dv, cfg.near_dup_threshold)
        Ytr = np.stack(splits[lang]["train"]["labels"].values) if lang in splits and len(splits[lang]["train"]) \
            else np.stack(all_train["labels"].values)
        Ydv = np.stack(dv["labels"].values)
        prior = (Ytr.mean(0) >= 0.5).astype(int)
        rows.append(dict(
            set=name, units=len(dv), articles=dv["article_id"].nunique(),
            article_overlap=len(set(all_train["article_id"]) & set(dv["article_id"])),
            docname_overlap=len(set(all_train["file"]) & set(dv["file"])),
            exact_text=f"{exact.mean():.1%}", near_dup=f"{(sim >= cfg.near_dup_threshold).mean():.1%}",
            median_max_sim=round(float(np.median(sim)), 3),
            empty=f"{(Ydv.sum(1) == 0).mean():.1%}" if not name.endswith('/predict') else 'unlabelled',
            majority_microF1=round(_official_micro(Ydv, np.tile(prior, (len(Ydv), 1))), 3) if not name.endswith('/predict') else '-',
        ))
    for l, d in splits.items():
        print(f"{l}: dev = {d['dev_source']}")
    if SMOKE_TEST:
        print("NOTE: synthetic smoke data has keyword cues per label; high dev F1 there is expected, not leakage.")
    return pd.DataFrame(rows).set_index("set")


def remove_dev_overlap(splits, cfg: Config) -> None:
    """Drop dev units that (near-)duplicate ANY unit any training phase sees, in any language."""
    all_train = pooled(splits, "train")
    for lang, d in splits.items():
        if not len(d["dev"]):
            continue
        exact, sim = train_dev_overlap(all_train, d["dev"], cfg.near_dup_threshold)
        bad = exact | (sim >= cfg.near_dup_threshold)
        if bad.any():
            print(f"[leak-filter] {lang}: dropping {int(bad.sum())}/{len(bad)} dev units duplicated in train")
        d["dev"] = d["dev"][~bad].reset_index(drop=True)

### Synthetic data (smoke test only)
Mirrors the release: same folder names and span-file names, CRLF documents, empty label fields, a SemEval-2023
style folder without empty paragraphs, a dev release with participant + 2b gold files, and a few republished
duplicate articles to exercise the audit. Labels are signalled by noisy cue words.

In [ ]:
def make_synthetic_release(root: str, seed: int = 0) -> None:
    rng = np.random.default_rng(seed)
    cue = {l: [f"{l.split('_')[0].lower()}{k}" for k in range(3)] for l in LABELS}
    filler = [f"w{k}" for k in range(80)]

    def article(n_par, allow_empty):
        body, spans = "Title line\r\n\r\n", []
        for _ in range(n_par):
            k = int(rng.choice([0, 1, 1, 2, 3])) if allow_empty else int(rng.choice([1, 1, 2, 3]))
            labs = sorted(rng.choice(LABELS, size=k, replace=False)) if k else []
            words = list(rng.choice(filler, size=12))
            for l in labs:
                if rng.random() < 0.8:
                    words += list(rng.choice(cue[l], size=2))
            if rng.random() < 0.3:                                  # distractor cue
                words.append(str(rng.choice(cue[str(rng.choice(LABELS))])))
            rng.shuffle(words)
            s = len(body); body += " ".join(words) + "."; spans.append((s, len(body), labs)); body += "\r\n\r\n"
        return body, spans

    def write_set(d, prefix, n_docs, kinds, allow_empty=True, n_dups=0):
        os.makedirs(os.path.join(d, "raw-documents"), exist_ok=True)
        lines = {k: [] for k in kinds}
        docs = []
        for a in range(n_docs):
            docs.append((f"{prefix}_{a}.txt",) + article(int(rng.integers(3, 8)), allow_empty))
        for a in range(n_dups):
            docs.append((f"{prefix}_{a}_republished.txt", docs[a][1], docs[a][2]))
        for fname, body, spans in docs:
            with open(os.path.join(d, "raw-documents", fname), "w", encoding="utf-8", newline="") as f:
                f.write(body)
            for s, e, labs in spans:
                for k in kinds:
                    lab = labs if k != "subtask-1-participant.txt" else None
                    lines[k].append("\t".join([fname, str(s), str(e)] + (lab if lab else [""] if lab is not None else [])))
        for k, ls in lines.items():
            with open(os.path.join(d, k), "w", encoding="utf-8", newline="") as f:
                f.write("\r\n".join(ls) + "\r\n")

    write_set(os.path.join(root, "1_initial-training-data/subtask1/SEMEVAL23_FRAMING_INITIAL_TRAIN-A/DE"),
              "DE_TRAIN_doc", 15, ["subtask-1-annotations.txt"], allow_empty=False)
    for lang in ["BG", "EN", "PT"]:
        write_set(os.path.join(root, f"2_training-second-batch-oct-revised/{lang}"), f"{lang}_M_TRAIN",
                  20, ["subtask-1-annotations.txt"], n_dups=2)
        dv = os.path.join(root, f"3_dev-set-nov-release/{lang}")
        write_set(dv, f"{lang}_M_DEV", 6, ["subtask-1-participant.txt"])
        rng_state = rng.bit_generator.state                         # 2b docs: separate files in the same folder
        tmp = os.path.join(root, "_tmp2b", lang)
        write_set(tmp, f"{lang}_M_DEV2B", 6, ["subtask-1-annotations-for-subtask-2b-participant.txt"])
        for fn in os.listdir(os.path.join(tmp, "raw-documents")):
            os.replace(os.path.join(tmp, "raw-documents", fn), os.path.join(dv, "raw-documents", fn))
        os.replace(os.path.join(tmp, "subtask-1-annotations-for-subtask-2b-participant.txt"),
                   os.path.join(dv, "subtask-1-annotations-for-subtask-2b-participant.txt"))
    import shutil; shutil.rmtree(os.path.join(root, "_tmp2b"), ignore_errors=True)
    write_set(os.path.join(root, "4_training-russian-nov/RU"), "RU_M_TRAIN_doc", 25, ["subtask-1-annotations.txt"], n_dups=2)


if SMOKE_TEST:
    import shutil
    shutil.rmtree(CFG.data_dir, ignore_errors=True)
    make_synthetic_release(CFG.data_dir)

In [ ]:
SOURCES = discover(CFG)
for k, v in SOURCES.items():
    print(f"{k:>9}: " + ", ".join(f"{s['lang']}({s['origin']})" for s in v))
assert SOURCES["train"], f"no subtask-1-annotations.txt found under {CFG.data_dir}"

SPLITS = build_splits(CFG, SOURCES)
TARGETS = [l for l in CFG.languages if l in SPLITS and len(SPLITS[l]["train"])]
LANGS_2027 = list(TARGETS)                       # all 2027 training languages (used as extra phase-2 data for EN)
if EN_ONLY:
    TARGETS = ["EN"]
PREDICT_SETS = {f"{s['lang']}/predict": load_span_file(s, labelled=False) for s in SOURCES["predict"]}
print("2027 training languages:", LANGS_2027, "| models built for:", TARGETS)

print("\nBEFORE leak filter"); print(leakage_audit(SPLITS, CFG, PREDICT_SETS).to_string())
if CFG.drop_dev_overlap:
    remove_dev_overlap(SPLITS, CFG)
    print("\nAFTER leak filter"); print(leakage_audit(SPLITS, CFG).to_string())
print()
for lang, d in SPLITS.items():
    for part in ["train", "dev", "extra"]:
        if len(d[part]):
            describe(d[part], f"{lang}/{part}")
describe(pooled(SPLITS, "train"), "POOLED phase-1 train")
describe(pooled(SPLITS, "dev"), "POOLED dev")

for lang, d in SPLITS.items():
    assert not set(pooled(SPLITS, "train")["article_id"]) & set(d["dev"]["article_id"]), f"article leak in {lang}"

    train: DE(2023), FR(2023), IT(2023), PL(2023), RU(2023), DE(2023), EN(2023), ES(2023), FR(2023), GR(2023), IT(2023), KA(2023), PL(2023), RU(2023), BG(2027), EN(2027), PT(2027), RU(2027)
 gold_dev: BG(2027), EN(2027), PT(2027)
  predict: BG(2027), EN(2027), PT(2027)
2027 training languages: ['BG', 'EN', 'PT', 'RU'] | models built for: ['EN']

BEFORE leak filter
BG: dev = gold ST1 labels of dev-release docs (2b file)
DE: dev = none (pre-training language)
EN: dev = gold ST1 labels of dev-release docs (2b file)
ES: dev = none (pre-training language)
FR: dev = none (pre-training language)
GR: dev = none (pre-training language)
IT: dev = none (pre-training language)
KA: dev = none (pre-training language)
PL: dev = none (pre-training language)
PT: dev = gold ST1 labels of dev-release docs (2b file)
RU: dev = article-grouped 20% split of 2027 train
            units  articles  article_overlap  docname_overlap exact_text near_dup  median_max_sim       empty majority_microF1
set            

## 4. Model
Encoder `paraphrase-multilingual-MiniLM-L12-v2` (384-d), **mean pooling** over non-pad tokens,
**no** L2 normalisation of the embedding (paper §3, reference `_mean_pooling`).
Head (paper): `Linear(384→256) → ReLU → Dropout(0.5) → Linear(256→14)`; sigmoid only at inference.

In [ ]:
HF_TOKEN = ""   # public models need no token. Only set this (a *valid* token string) for private/gated models.


def hf_load(name: str):
    """Load tokenizer + encoder anonymously (token=False), so an expired saved token cannot cause 401 errors."""
    for k in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        if HF_TOKEN is None and os.environ.pop(k, None) is not None:
            print(f"[info] ignoring {k} from the environment for public model downloads")
    tok_arg = HF_TOKEN if HF_TOKEN else False
    return (AutoTokenizer.from_pretrained(name, token=tok_arg),
            AutoModel.from_pretrained(name, token=tok_arg))


def build_tokenizer_and_encoder(cfg: Config):
    if SMOKE_TEST:
        from transformers import BertConfig, BertModel, BertTokenizerFast
        words = set()
        for d in SPLITS.values():
            for t in pd.concat([d["train"], d["dev"], d["extra"]])["text"]:
                words.update(re.findall(r"\w+|[^\w\s]", t.lower()))
        vocab = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"] + sorted(words)
        vpath = os.path.join(cfg.out_dir, "smoke_vocab.txt")
        with open(vpath, "w") as f:
            f.write("\n".join(vocab))
        tok = BertTokenizerFast(vocab_file=vpath, do_lower_case=True, model_max_length=128)
        enc = BertModel(BertConfig(vocab_size=len(vocab), hidden_size=384, num_hidden_layers=2,
                                   num_attention_heads=4, intermediate_size=256, max_position_embeddings=128),
                        add_pooling_layer=False)
        return tok, enc
    return hf_load(cfg.model_name)


def mean_pooling(token_embeddings: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    m = attention_mask.unsqueeze(-1).to(token_embeddings.dtype)
    return (token_embeddings * m).sum(1) / m.sum(1).clamp(min=1e-9)


class Encoder(nn.Module):
    def __init__(self, hf_model: nn.Module):
        super().__init__()
        self.model = hf_model
        self.dim = hf_model.config.hidden_size

    def forward(self, input_ids, attention_mask):
        out = self.model(input_ids=input_ids, attention_mask=attention_mask)
        return mean_pooling(out[0], attention_mask)   # no normalisation


def build_head(cfg: Config, in_dim: int) -> nn.Module:
    if cfg.head_variant == "paper":
        return nn.Sequential(nn.Linear(in_dim, cfg.head_hidden), nn.ReLU(), nn.Dropout(cfg.head_dropout),
                             nn.Linear(cfg.head_hidden, NUM_CLASSES))
    if cfg.head_variant == "reference_code":
        return nn.Sequential(nn.Linear(in_dim, 256), nn.Dropout(), nn.ReLU(),
                             nn.Linear(256, 256), nn.Dropout(), nn.ReLU(),
                             nn.Linear(256, NUM_CLASSES), nn.Dropout())
    raise ValueError(cfg.head_variant)


TOKENIZER, _ENC0 = build_tokenizer_and_encoder(CFG)
MAX_LEN = CFG.max_length or min(int(TOKENIZER.model_max_length), 512,
                                int(getattr(_ENC0.config, "max_position_embeddings", 512)))
INIT_ENCODER_STATE = copy.deepcopy(_ENC0.state_dict())
CURRENT_BACKBONE = CFG.model_name


def use_backbone(name: str) -> None:
    """Swap the global tokenizer/encoder used by fresh_encoder() and collate(). No-op in smoke mode."""
    global TOKENIZER, _ENC0, MAX_LEN, INIT_ENCODER_STATE, CURRENT_BACKBONE
    if SMOKE_TEST or name == CURRENT_BACKBONE:
        return
    TOKENIZER, _ENC0 = hf_load(name)
    MAX_LEN = CFG.max_length or min(int(TOKENIZER.model_max_length), 512,
                                    int(getattr(_ENC0.config, "max_position_embeddings", 512)))
    INIT_ENCODER_STATE = copy.deepcopy(_ENC0.state_dict())
    CURRENT_BACKBONE = name
    print("backbone ->", name, "| dim", _ENC0.config.hidden_size, "| max_len", MAX_LEN)
print("embedding dim:", _ENC0.config.hidden_size, "| max_length:", MAX_LEN)
print(build_head(CFG, _ENC0.config.hidden_size))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

embedding dim: 384 | max_length: 512
Sequential(
  (0): Linear(in_features=384, out_features=256, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=256, out_features=14, bias=True)
)


In [ ]:
class UnitDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.texts = df["text"].tolist()
        self.labels = torch.tensor(np.stack(df["labels"].values), dtype=torch.float32) if len(df) else \
            torch.zeros((0, NUM_CLASSES))

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        return self.texts[i], self.labels[i]


def collate(batch):
    texts, labels = zip(*batch)
    enc = TOKENIZER(list(texts), truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")
    return enc["input_ids"], enc["attention_mask"], torch.stack(labels)

## 5. Label-aware contrastive loss (HeroCon-style, paper Eq. 2–3)

For each class $c$ with positives $\mathcal P(c)$ and negatives $\mathcal N(c)$ in the batch:

$$\sigma_{ij}=1-\frac{d(Y_i,Y_j)}{|C|},\qquad \gamma_{ik}=d(Y_i,Y_k),\qquad
\delta_{ij}=\frac{\sigma_{ij}f(i,j)+\sum_{k\in\mathcal N(c)}\gamma_{ik}f(i,k)}{|\mathcal N(c)|+1}$$
$$\mathcal L_c=-\mathbb E_{i,j\in\mathcal P(c)}\log\frac{\sigma_{ij}f(i,j)}{\delta_{ij}},\qquad
\mathcal L_{CON}=\tfrac1{|C|}\sum_c\mathcal L_c$$

**What the reference code does that the paper text leaves out** (checked in `custom_loss.py`):
* $f(i,j)=\exp(\cos(i,j)/T)$ with $T=1$, not the raw cosine. A raw cosine can be ≤0, making the log
  undefined, so the exponential is required. We keep it.
* Self-pairs $i=j$ are included and the per-class losses are **summed**; the paper says mean. Defaults here
  follow the paper and your spec (mean, $i\neq j$, skip classes with <2 positives); set
  `con_include_self_pairs=True, con_class_reduction="sum", con_min_positives=1` to reproduce the code exactly
  (a unit test checks this equivalence).

Numerics: computed in log-space, $\log(\sigma f)=\log\sigma+\cos/T$, the denominator via `logsumexp`.
$\sigma_{ij}>0$ always (two positives of $c$ agree on $c$, so $d\le |C|-1$) and $\gamma_{ik}\ge1$ (they disagree
on $c$), so no log of zero. Note that $\mathcal L_c$ can be negative, as in the reference.

In [ ]:
def hamming_matrix(y: torch.Tensor) -> torch.Tensor:
    return y @ (1 - y).T + (1 - y) @ y.T


class HeroConLoss(nn.Module):
    def __init__(self, num_classes: int, temperature: float = 1.0, include_self_pairs: bool = False,
                 class_reduction: str = "mean", min_positives: int = 2, eps: float = 1e-8):
        super().__init__()
        assert class_reduction in ("mean", "sum")
        self.C, self.T, self.self_pairs = num_classes, temperature, include_self_pairs
        self.red, self.min_pos, self.eps = class_reduction, min_positives, eps

    def forward(self, z: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        z = z.float(); y = y.float()
        zn = z / z.norm(dim=1, keepdim=True).clamp_min(self.eps)
        logf = (zn @ zn.T) / self.T                       # log f(i,j) = cos/T
        d = hamming_matrix(y)
        losses = []
        for c in range(self.C):
            pos = (y[:, c] == 1)
            n_pos = int(pos.sum())
            if n_pos < max(self.min_pos, 1) or (not self.self_pairs and n_pos < 2):
                continue
            neg = ~pos
            n_neg = int(neg.sum())
            P = pos.nonzero(as_tuple=True)[0]; N = neg.nonzero(as_tuple=True)[0]
            sigma = (1.0 - d[P][:, P] / self.C).clamp_min(self.eps)
            log_num = sigma.log() + logf[P][:, P]                       # (p, p): log sigma_ij f(i,j)
            if n_neg > 0:
                gamma = d[P][:, N].clamp_min(self.eps)
                log_negsum = torch.logsumexp(gamma.log() + logf[P][:, N], dim=1, keepdim=True)  # (p, 1)
                log_delta = torch.logaddexp(log_num, log_negsum) - math.log(n_neg + 1)
            else:
                log_delta = log_num                                     # delta = sigma f / 1
            term = log_delta - log_num                                  # = -log(sigma f / delta)
            if not self.self_pairs:
                term = term[~torch.eye(n_pos, dtype=torch.bool, device=z.device)]
            losses.append(term.mean())
        if not losses:
            return z.sum() * 0.0                                        # keeps graph, finite zero
        L = torch.stack(losses)
        return L.mean() if self.red == "mean" else L.sum()


def make_con_loss(cfg: Config) -> HeroConLoss:
    return HeroConLoss(NUM_CLASSES, cfg.con_temperature, cfg.con_include_self_pairs,
                       cfg.con_class_reduction, cfg.con_min_positives, cfg.con_eps)


class ReferenceWCSL(nn.Module):
    """Verbatim port of WeightedCosineSimilarityLoss from the reference repo (for the equivalence test only)."""
    def __init__(self, class_num):
        super().__init__(); self.class_num = class_num

    def forward(self, features, labels):
        loss = torch.tensor(0, device=features.device, dtype=torch.float64)
        for i in range(self.class_num):
            pos_idx = torch.where(labels[:, i] == 1)[0]
            if len(pos_idx) == 0:
                continue
            neg_idx = torch.where(labels[:, i] != 1)[0]
            pos_s, neg_s = features[pos_idx, :], features[neg_idx, :]
            size = neg_s.shape[0] + 1
            dist = hamming_matrix(labels)
            pos_w = 1 - dist[pos_idx, :][:, pos_idx] / self.class_num
            neg_w = dist[pos_idx, :][:, neg_idx]
            pos_dis = self._ecs(pos_s, pos_s) * pos_w
            neg_dis = self._ecs(pos_s, neg_s) * neg_w
            loss += torch.mean(torch.log((neg_dis.sum(1) + pos_dis) / (pos_dis * size)))
        return loss

    @staticmethod
    def _ecs(x1, x2, eps=1e-15):
        w1, w2 = x1.norm(p=2, dim=1, keepdim=True), x2.norm(p=2, dim=1, keepdim=True)
        return torch.exp(x1 @ x2.t() / (w1 * w2.t()).clamp(min=eps))

## 6. Contrast sampler (paper's "CS" variant)
Port of the reference `ContrastSampler`: each batch first draws `min_samples_from_class` units for every
class (least-used first), then fills up with the least-used remaining units; an epoch ends once every unit
has been seen. The reference raises an error if a class has fewer than `min_samples_from_class` examples
in the whole set; with small target-language sets that happens, so **ASSUMPTION** here such classes are
simply skipped. With batch size 26 and 14 classes the paper's "at least one per class" means
`min_samples_from_class=1` (the reference default 2 would need batch ≥ 28).

In [ ]:
class ContrastSampler(Sampler[List[int]]):
    def __init__(self, labels: torch.Tensor, batch_size: int, min_samples_from_class: int = 1, seed: int = 42):
        self.L = labels.clone().detach().t()             # (C, n)
        self.n, self.bs, self.k = labels.shape[0], batch_size, min_samples_from_class
        self.class_idxs = [torch.argwhere(self.L[i]).flatten() for i in range(self.L.shape[0])]
        self.active = [i for i, ix in enumerate(self.class_idxs) if len(ix) >= self.k]
        if batch_size < len(self.active) * self.k:
            raise ValueError("batch_size must be >= num_active_classes * min_samples_from_class")
        self.g = torch.Generator(); self.g.manual_seed(seed)
        self.used = torch.zeros(self.n)

    def _least_used(self, in_batch, idxs, k):
        uses = self.used[idxs] + in_batch[idxs]
        m = torch.min(uses)
        cand = torch.argwhere(uses == m).flatten()
        if len(cand) < k:
            cand = torch.cat([cand, torch.argwhere(uses == m + 1).flatten()])
        return cand

    def __iter__(self):
        self.used = torch.zeros(self.n)
        while (self.used == 0).any():
            batch, in_batch = [], torch.zeros(self.n)
            required = torch.zeros(self.L.shape[0]); required[self.active] = self.k
            for c in torch.randperm(self.L.shape[0], generator=self.g).tolist():
                need = int(required[c])
                if need < 1:
                    continue
                cand = self._least_used(in_batch, self.class_idxs[c], need)
                sel = self.class_idxs[c][cand[torch.randperm(len(cand), generator=self.g)[:need]]]
                self.used[sel] += 1; in_batch[sel] = 2 ** 20
                batch.append(sel)
                required -= self.L.t()[sel].sum(0)
            samples = torch.cat(batch).tolist() if batch else []
            rest = self.bs - len(samples)
            if rest > 0:
                cand = self._least_used(in_batch, torch.arange(self.n), rest)
                cand = cand[in_batch[cand] == 0]
                sel = cand[torch.randperm(len(cand), generator=self.g)[:rest]]
                self.used[sel] += 1
                samples += sel.tolist()
            yield samples

    def __len__(self):
        return max(1, math.ceil(self.n / self.bs))

## 7. Trainer
Stages, as in the reference `Trainer`:
* **head stage** (`train_head`): encoder in `eval()` and frozen, embeddings computed once, head trained with
  BCE only on the cached embeddings.
* **contrastive stage** (`train_joint`): encoder + head trained together on $\mathcal L_{BCE}+\alpha\mathcal L_{CON}$,
  separate AdamW optimisers (head 1e-3, encoder 2e-5), per-epoch ExponentialLR (reference code).

Every stage writes `log.json`; `save()` writes `config.json` and a checkpoint per phase.

In [ ]:
def encoder_fingerprint(enc: nn.Module) -> torch.Tensor:
    return torch.cat([p.detach().flatten().float().cpu() for p in enc.parameters()])


class MCPTTrainer:
    def __init__(self, cfg: Config, encoder: Encoder, head: Optional[nn.Module] = None, run_dir: str = "."):
        self.cfg, self.enc, self.run_dir = cfg, encoder.to(DEVICE), run_dir
        os.makedirs(run_dir, exist_ok=True)
        self.bce = nn.BCEWithLogitsLoss()
        self.con = make_con_loss(cfg)
        self.enc_opt = AdamW(self.enc.parameters(), lr=cfg.encoder_lr)
        self.enc_sched = ExponentialLR(self.enc_opt, gamma=cfg.encoder_gamma if cfg.use_lr_decay else 1.0)
        self.set_head(head if head is not None else build_head(cfg, self.enc.dim))
        self.log: List[dict] = []

    # ---- head management -------------------------------------------------------------------
    def set_head(self, head: nn.Module):
        self.head = head.to(DEVICE)
        self.head_opt = AdamW(self.head.parameters(), lr=self.cfg.head_lr)
        self.head_sched = ExponentialLR(self.head_opt, gamma=self.cfg.head_gamma if self.cfg.use_lr_decay else 1.0)

    def reinit_head(self, seed: int):
        torch.manual_seed(seed)
        self.set_head(build_head(self.cfg, self.enc.dim))

    # ---- embeddings ------------------------------------------------------------------------
    @torch.no_grad()
    def embed(self, df: pd.DataFrame) -> Tuple[torch.Tensor, torch.Tensor]:
        self.enc.eval()
        dl = DataLoader(UnitDataset(df), batch_size=self.cfg.eval_batch_size, shuffle=False,
                        collate_fn=collate, num_workers=self.cfg.num_workers)
        Z, Y = [], []
        for ids, am, y in dl:
            Z.append(self.enc(ids.to(DEVICE), am.to(DEVICE)).float().cpu()); Y.append(y)
        return torch.cat(Z), torch.cat(Y)

    @torch.no_grad()
    def predict_proba(self, df: pd.DataFrame) -> np.ndarray:
        Z, _ = self.embed(df)
        self.head.eval()
        out = [torch.sigmoid(self.head(Z[i:i + 1024].to(DEVICE))).cpu() for i in range(0, len(Z), 1024)]
        return torch.cat(out).numpy() if out else np.zeros((0, NUM_CLASSES))

    # ---- stages ----------------------------------------------------------------------------
    def _record(self, stage, epoch, **kw):
        rec = dict(stage=stage, epoch=epoch, time=time.time(), head_lr=self.head_opt.param_groups[0]["lr"],
                   enc_lr=self.enc_opt.param_groups[0]["lr"], **kw)
        self.log.append(rec)
        print({k: (float(f"{v:.4g}") if isinstance(v, float) else v) for k, v in rec.items() if k != "time"})

    def _maybe_eval(self, stage, epoch, n_epochs, dev_df):
        e = self.cfg.eval_every
        if dev_df is None or len(dev_df) == 0:
            return {}
        if (e and (epoch + 1) % e == 0) or epoch + 1 == n_epochs:
            P = self.predict_proba(dev_df)
            Y = np.stack(dev_df["labels"].values)
            g = dev_df["article_id"].astype(str).values
            thr, res = tune_threshold(Y, P, self.cfg)
            res.update(crossfit_scores(Y, P, g, self.cfg))
            out = {"dev_micro_f1": res["micro_f1"], "dev_macro_f1": res["macro_f1"], "dev_thr": thr,
                   **({"dev_cf_micro_f1": res["cf_micro_f1"]} if "cf_micro_f1" in res else {})}
            langs = dev_df["lang"].values
            if len(set(langs)) > 1:                      # pooled dev -> also report each language separately
                for l in sorted(set(langs)):
                    m = langs == l
                    cf = crossfit_scores(Y[m], P[m], g[m], self.cfg)
                    out[f"{l}_cf_micro_f1"] = cf.get("cf_micro_f1", tune_threshold(Y[m], P[m], self.cfg)[1]["micro_f1"])
            rl = REPORT_LANG
            en_score = out.get(f"{rl}_cf_micro_f1", out.get("dev_cf_micro_f1") if set(langs) == {rl} else None)
            if en_score is not None:
                print(f"  >> {stage} epoch {epoch}: {rl} cf micro-F1 = {en_score:.4f}")
            return out
        return {}

    @torch.no_grad()
    def _sel_score(self, Y: np.ndarray, P: np.ndarray) -> float:
        return tune_threshold(Y, P, self.cfg)[1]["micro_f1"]

    def train_head(self, df: pd.DataFrame, n_epochs: int, stage: str, dev_df: Optional[pd.DataFrame] = None,
                   select_df: Optional[pd.DataFrame] = None):
        """Encoder frozen (eval mode, cached embeddings), BCE only.
        select_df: keep the head from the epoch with the best micro-F1 on select_df (never the dev set)."""
        if n_epochs <= 0:
            return
        for p in self.enc.parameters():
            p.requires_grad_(False)
        Z, Y = self.embed(df)
        if select_df is not None:
            Zs, Ys = self.embed(select_df); Ys = Ys.numpy(); best = (-1.0, None, -1)
        g = torch.Generator(); g.manual_seed(self.cfg.seed)
        for ep in range(n_epochs):
            self.head.train()
            perm = torch.randperm(len(Z), generator=g)
            losses = []
            for i in range(0, len(Z), self.cfg.head_batch_size):
                b = perm[i:i + self.cfg.head_batch_size]
                loss = self.bce(self.head(Z[b].to(DEVICE)), Y[b].to(DEVICE))
                self.head_opt.zero_grad(set_to_none=True); loss.backward(); self.head_opt.step()
                losses.append(loss.item())
            self.head_sched.step()
            sel = {}
            if select_df is not None:
                self.head.eval()
                with torch.no_grad():
                    Ps = torch.sigmoid(self.head(Zs.to(DEVICE))).cpu().numpy()
                sc = self._sel_score(Ys, Ps); sel = {"sel_f1": sc}
                if sc > best[0]:
                    best = (sc, copy.deepcopy({k: v.cpu() for k, v in self.head.state_dict().items()}), ep)
            self._record(stage, ep, bce=float(np.mean(losses)), **sel, **self._maybe_eval(stage, ep, n_epochs, dev_df))
        if select_df is not None and best[1] is not None:
            self.head.load_state_dict(best[1]); print(f"  [{stage}] restored best epoch {best[2]} (sel_f1={best[0]:.4f})")
        for p in self.enc.parameters():
            p.requires_grad_(True)

    def _loader(self, df: pd.DataFrame, sampler: str) -> DataLoader:
        ds = UnitDataset(df)
        if sampler == "contrast":
            bs = ContrastSampler(ds.labels, self.cfg.batch_size, self.cfg.cs_min_samples_per_class, self.cfg.seed)
            return DataLoader(ds, batch_sampler=bs, collate_fn=collate, num_workers=self.cfg.num_workers)
        g = torch.Generator(); g.manual_seed(self.cfg.seed)
        return DataLoader(ds, sampler=RandomSampler(ds, generator=g), batch_size=self.cfg.batch_size,
                          collate_fn=collate, num_workers=self.cfg.num_workers)

    def train_contrastive(self, df: pd.DataFrame, n_epochs: int, stage: str, alpha: Optional[float] = None,
                          sampler: Optional[str] = None, dev_df: Optional[pd.DataFrame] = None,
                          select_df: Optional[pd.DataFrame] = None):
        """Encoder + head end-to-end on L_BCE + alpha * L_CON.
        select_df: keep encoder+head from the epoch with the best micro-F1 on select_df (never the dev set)."""
        if n_epochs <= 0:
            return
        best = (-1.0, None, -1)
        Ys = np.stack(select_df["labels"].values) if select_df is not None else None
        alpha = self.cfg.alpha if alpha is None else alpha
        dl = self._loader(df, sampler or self.cfg.sampler)
        for ep in range(n_epochs):
            self.enc.train(); self.head.train()
            lb, lc = [], []
            for ids, am, y in dl:
                ids, am, y = ids.to(DEVICE), am.to(DEVICE), y.to(DEVICE)
                z = self.enc(ids, am)
                l_bce = self.bce(self.head(z), y)
                l_con = self.con(z, y) if alpha > 0 else torch.zeros((), device=DEVICE)
                loss = l_bce + alpha * l_con
                if not torch.isfinite(loss):
                    raise FloatingPointError(f"non-finite loss at {stage} epoch {ep}")
                self.enc_opt.zero_grad(set_to_none=True); self.head_opt.zero_grad(set_to_none=True)
                loss.backward()
                self.enc_opt.step(); self.head_opt.step()
                lb.append(l_bce.item()); lc.append(l_con.item())
            self.enc_sched.step(); self.head_sched.step()
            sel = {}
            if select_df is not None:
                sc = self._sel_score(Ys, self.predict_proba(select_df)); sel = {"sel_f1": sc}
                if sc > best[0]:
                    best = (sc, ({k: v.detach().cpu().clone() for k, v in self.enc.state_dict().items()},
                                 {k: v.detach().cpu().clone() for k, v in self.head.state_dict().items()}), ep)
            self._record(stage, ep, bce=float(np.mean(lb)), con=float(np.mean(lc)), **sel,
                         **self._maybe_eval(stage, ep, n_epochs, dev_df))
        if select_df is not None and best[1] is not None:
            self.enc.load_state_dict(best[1][0]); self.head.load_state_dict(best[1][1])
            print(f"  [{stage}] restored best epoch {best[2]} (sel_f1={best[0]:.4f})")

    # ---- persistence -----------------------------------------------------------------------
    def save(self, name: str, extra: Optional[dict] = None):
        d = os.path.join(self.run_dir, name)
        os.makedirs(d, exist_ok=True)
        with open(os.path.join(d, "config.json"), "w") as f:
            json.dump(asdict(self.cfg), f, indent=1, default=str)
        with open(os.path.join(d, "log.json"), "w") as f:
            json.dump(self.log, f, indent=1, default=float)
        ck = {"head": self.head.state_dict(), "head_opt": self.head_opt.state_dict(),
              "head_sched": self.head_sched.state_dict(), "extra": extra or {}}
        if self.cfg.save_encoder_in_checkpoints:
            ck.update(encoder=self.enc.state_dict(), enc_opt=self.enc_opt.state_dict(),
                      enc_sched=self.enc_sched.state_dict())
        torch.save(ck, os.path.join(d, "checkpoint.pt"))
        return d

    def load(self, path: str, encoder=True, head=True, enc_optim=True):
        ck = torch.load(os.path.join(path, "checkpoint.pt"), map_location=DEVICE)
        if encoder and "encoder" in ck:
            self.enc.load_state_dict(ck["encoder"])
            if enc_optim:
                self.enc_opt.load_state_dict(ck["enc_opt"]); self.enc_sched.load_state_dict(ck["enc_sched"])
        if head:
            self.head.load_state_dict(ck["head"])
            self.head_opt.load_state_dict(ck["head_opt"]); self.head_sched.load_state_dict(ck["head_sched"])


def fresh_encoder(state: Optional[dict] = None) -> Encoder:
    enc = copy.deepcopy(_ENC0)
    enc.load_state_dict(state if state is not None else INIT_ENCODER_STATE)
    return Encoder(enc)

## 8. Evaluation & export
* Threshold: one global threshold, chosen on the **dev** split by micro-F1 (grid 0.05–0.95). Never on test.
* Metrics: (a) 14-class micro/macro-F1 (sklearn); (b) **official-style** micro/macro-F1, where an empty label set
  counts as the label `None` (task page: *"we use the label None to represent no framing"*), i.e. 15 classes.
  ASSUMPTION: this mirrors the official scorer; if you have the scorer script, set `official_scorer_cmd`
  and it is called on the exported files as well.
* Export: `file-name \t start \t end \t label \t label ...`, empty field for no framing.

In [ ]:
SCORING_LABELS = LABELS + ["None"]


def to_official_matrix(Y: np.ndarray) -> np.ndarray:
    Y = (np.asarray(Y) >= 0.5).astype(int)
    return np.concatenate([Y, (Y.sum(1) == 0).astype(int)[:, None]], axis=1)


def binarize(P: np.ndarray, thr: float, force_one: bool) -> np.ndarray:
    Yh = (P >= thr).astype(int)
    if force_one and len(P):
        empty = Yh.sum(1) == 0
        Yh[empty, P[empty].argmax(1)] = 1
    return Yh


def scores(Y: np.ndarray, Yh: np.ndarray) -> dict:
    Y, Yh = (np.asarray(Y) >= 0.5).astype(int), np.asarray(Yh).astype(int)
    O, Oh = to_official_matrix(Y), to_official_matrix(Yh)
    return dict(
        micro_f1=f1_score(O, Oh, average="micro", zero_division=0),
        macro_f1=f1_score(O, Oh, average="macro", zero_division=0),
        micro_f1_14=f1_score(Y, Yh, average="micro", zero_division=0),
        macro_f1_14=f1_score(Y, Yh, average="macro", zero_division=0),
    )


def tune_threshold(Y: np.ndarray, P: np.ndarray, cfg: Config) -> Tuple[float, dict]:
    best = (0.5, None)
    for t in cfg.threshold_grid:
        s = scores(Y, binarize(P, t, cfg.force_at_least_one))
        if best[1] is None or s["micro_f1"] > best[1]["micro_f1"]:
            best = (float(t), s)
    return best


def crossfit_scores(Y: np.ndarray, P: np.ndarray, groups: np.ndarray, cfg: Config) -> dict:
    """Honest estimate: split dev articles in two halves, tune the threshold on one half and score the other
    (and vice versa). The plain tuned-on-dev score is optimistic because threshold and score share data."""
    ug = np.unique(groups)
    if len(ug) < 4:
        return {}
    half = set(np.random.default_rng(cfg.seed).permutation(ug)[: len(ug) // 2])
    m = np.array([g in half for g in groups])
    Yh = np.zeros_like(Y, dtype=int)
    for tune, apply in [(m, ~m), (~m, m)]:
        t, _ = tune_threshold(Y[tune], P[tune], cfg)
        Yh[apply] = binarize(P[apply], t, cfg.force_at_least_one)
    s = scores(Y, Yh)
    return {"cf_micro_f1": s["micro_f1"], "cf_macro_f1": s["macro_f1"]}


def evaluate(trainer: MCPTTrainer, dev_df: pd.DataFrame, cfg: Config, threshold: Optional[float] = None) -> dict:
    P = trainer.predict_proba(dev_df)
    Y = np.stack(dev_df["labels"].values)
    if threshold is None:
        thr, s = tune_threshold(Y, P, cfg)
        s.update(crossfit_scores(Y, P, dev_df["article_id"].astype(str).values, cfg))
    else:
        thr, s = threshold, scores(Y, binarize(P, threshold, cfg.force_at_least_one))
    return dict(threshold=thr, **s)


def write_submission(df: pd.DataFrame, Yh: np.ndarray, path: str, cfg: Config) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for (_, r), y in zip(df.iterrows(), Yh):
            labs = vec_to_labels(y)
            if labs:
                f.write("\t".join([str(r["file"]), str(r["start"]), str(r["end"])] + labs) + "\n")
            else:
                f.write(f"{r['file']}\t{r['start']}\t{r['end']}" + ("\t" if cfg.empty_label_trailing_tab else "") + "\n")


def read_submission(path: str) -> List[Tuple[str, int, int, List[str]]]:
    return _parse_span_file(path, with_labels=True)


def run_official_scorer(gold_path: str, pred_path: str, cfg: Config) -> Optional[str]:
    if not cfg.official_scorer_cmd:
        return None
    cmd = cfg.official_scorer_cmd.format(gold=gold_path, pred=pred_path)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout, r.stderr)
    return r.stdout

## 9. Unit tests
Run before any long training. They use the configured encoder (tiny in smoke mode, MiniLM otherwise).

In [ ]:
def test_loss_finite_and_matches_reference():
    torch.manual_seed(0)
    con = HeroConLoss(NUM_CLASSES)
    # imbalanced batch: rare labels, all-zero rows, a class present in every row (no negatives),
    # a class with a single positive, duplicate and zero embeddings
    y = torch.zeros(26, NUM_CLASSES)
    y[:, 0] = 1                     # every row positive for class 0 -> N(c) empty
    y[0, 1] = 1                     # single positive -> skipped
    y[[1, 2], 2] = 1                # rare pair
    y[3:6, 3] = 1
    y[10:, 0] = 0                   # rows 10.. all-zero
    z = torch.randn(26, 384, requires_grad=True)
    with torch.no_grad():
        z[4] = z[5]                 # identical embeddings
        z[7] = 0                    # zero vector
    L = con(z, y)
    assert torch.isfinite(L), L
    L.backward()
    assert torch.isfinite(z.grad).all()
    # no class with >=2 positives -> 0, finite, differentiable
    y2 = torch.zeros(8, NUM_CLASSES); y2[0, 5] = 1
    z2 = torch.randn(8, 384, requires_grad=True)
    L2 = con(z2, y2); L2.backward()
    assert torch.isfinite(L2) and L2.item() == 0.0
    # large-norm embeddings and tiny temperature stay finite in log-space
    L3 = HeroConLoss(NUM_CLASSES, temperature=0.01)(torch.randn(26, 384) * 1e4, (torch.rand(26, NUM_CLASSES) < .2).float())
    assert torch.isfinite(L3)
    # exact equivalence with the reference implementation in reference mode
    for trial in range(5):
        yy = (torch.rand(26, NUM_CLASSES) < 0.25).double()
        zz = torch.randn(26, 384, dtype=torch.float64)
        ref = ReferenceWCSL(NUM_CLASSES)(zz, yy)
        ours = HeroConLoss(NUM_CLASSES, include_self_pairs=True, class_reduction="sum", min_positives=1)(zz, yy)
        assert torch.allclose(ours.double(), ref, rtol=1e-4, atol=1e-5), (ours, ref)
    # brute-force check of the paper formula (mean over classes, i != j)
    yy = (torch.rand(12, NUM_CLASSES) < 0.3).float(); zz = torch.randn(12, 384)
    zn = F.normalize(zz, dim=1); f = torch.exp(zn @ zn.T); d = hamming_matrix(yy); per = []
    for c in range(NUM_CLASSES):
        P = [i for i in range(12) if yy[i, c] == 1]; N = [k for k in range(12) if yy[k, c] == 0]
        if len(P) < 2:
            continue
        t = []
        for i in P:
            for j in P:
                if i == j:
                    continue
                s = 1 - d[i, j] / NUM_CLASSES
                delta = (s * f[i, j] + sum(d[i, k] * f[i, k] for k in N)) / (len(N) + 1)
                t.append(-torch.log(s * f[i, j] / delta))
        per.append(torch.stack(t).mean())
    assert torch.allclose(HeroConLoss(NUM_CLASSES)(zz, yy), torch.stack(per).mean(), atol=1e-5)


def test_encoder_frozen_then_trained():
    cfg = copy.deepcopy(CFG); cfg.eval_every = 0; cfg.save_encoder_in_checkpoints = False
    df = pooled(SPLITS, "train").head(60)
    tr = MCPTTrainer(cfg, fresh_encoder(), run_dir=os.path.join(cfg.out_dir, "_test"))
    before = encoder_fingerprint(tr.enc.model)
    h_before = encoder_fingerprint(tr.head)
    tr.train_head(df, 1, "test_head")
    assert torch.equal(before, encoder_fingerprint(tr.enc.model)), "encoder changed in head-only stage"
    assert not torch.equal(h_before, encoder_fingerprint(tr.head)), "head did not train"
    assert all(p.requires_grad for p in tr.enc.parameters()), "requires_grad not restored"
    tr.train_contrastive(df, 1, "test_con")
    assert not torch.equal(before, encoder_fingerprint(tr.enc.model)), "encoder unchanged in contrastive stage"
    mid = encoder_fingerprint(tr.enc.model)
    tr.reinit_head(1); tr.train_head(df, 1, "test_post")
    assert torch.equal(mid, encoder_fingerprint(tr.enc.model)), "encoder changed in post-training"


def test_contrastive_loss_finite_in_training_batches():
    # every real training batch (imbalanced, rare labels) gives a finite loss, both samplers
    df = pooled(SPLITS, "train")
    con = make_con_loss(CFG)
    for smp in ["random", "contrast"]:
        tr = MCPTTrainer(CFG, fresh_encoder(), run_dir=os.path.join(CFG.out_dir, "_test"))
        tr.enc.eval()
        with torch.no_grad():
            for b, (ids, am, y) in enumerate(tr._loader(df, smp)):
                assert torch.isfinite(con(tr.enc(ids.to(DEVICE), am.to(DEVICE)), y.to(DEVICE)))
                if b > 20:
                    break


def test_contrast_sampler_covers_classes():
    df = pooled(SPLITS, "train")
    labels = UnitDataset(df).labels
    s = ContrastSampler(labels, 26, 1, seed=0)
    seen = set()
    for batch in s:
        assert len(batch) == len(set(batch)) and len(batch) <= 26
        present = labels[batch].sum(0) > 0
        for c in s.active:
            assert present[c], f"class {LABELS[c]} missing from batch"
        seen.update(batch)
    assert seen == set(range(len(labels)))


def test_export_format_roundtrip():
    df = pd.DataFrame(dict(file=["a.txt", "a.txt", "b.txt"], start=[0, 10, 0], end=[9, 20, 5],
                           text=["x", "y", "z"], labels=[np.zeros(14)] * 3))
    Yh = np.zeros((3, NUM_CLASSES), int); Yh[0, LABEL2ID["Economic"]] = 1
    Yh[1, LABEL2ID["Political"]] = Yh[1, LABEL2ID["Morality"]] = 1        # row 2 stays empty
    p = os.path.join(CFG.out_dir, "_test", "sub.txt")
    cfg = copy.deepcopy(CFG)
    write_submission(df, Yh, p, cfg)
    lines = open(p, encoding="utf-8").read().split("\n")
    assert lines[0] == "a.txt\t0\t9\tEconomic"
    assert lines[1] == "a.txt\t10\t20\tMorality\tPolitical"
    assert lines[2] == "b.txt\t0\t5\t" and lines[2].split("\t")[3] == ""
    back = read_submission(p)
    assert [r[3] for r in back] == [["Economic"], ["Morality", "Political"], []]
    assert len(back) == 3
    # empty-label units are scored as "None"
    s = scores(np.zeros((2, 14)), np.zeros((2, 14)))
    assert s["micro_f1"] == 1.0


def test_labels_normalise():
    assert normalise_label("Capacity and resurces") == "Capacity_and_resources"
    assert normalise_label("Legality, constitutionality and jurisprudence") == "Legality_Constitutionality_and_jurisprudence"
    assert labels_to_vec([]).sum() == 0


def test_group_split_no_leak():
    for lang, d in SPLITS.items():
        assert not set(d["train"]["article_id"]) & set(d["dev"]["article_id"])


def test_duplicate_articles_same_split():
    cfg = copy.deepcopy(CFG); cfg.dedup_aware_split = True
    rows = []
    for a in range(40):
        for p in range(3):
            rows.append(dict(lang="xx", article_id=f"a{a}", file=f"a{a}", start=p, end=p + 1,
                             text=f"unique paragraph number {p} of article {a} with enough characters",
                             labels=np.zeros(NUM_CLASSES, np.float32)))
    # a17..a19 are republished copies of a3; a25 shares one paragraph with a30
    for dup in ["a17", "a18", "a19"]:
        for r in rows:
            if r["article_id"] == dup:
                r["text"] = r["text"].replace(f"article {dup[1:]} ", "article 3 ")
    rows.append(dict(rows[-1], article_id="a25", text="shared long quoted paragraph appearing in two articles"))
    rows.append(dict(rows[-1], article_id="a30"))
    rows.append(dict(rows[-1], article_id="a31", text="Photo: Reuters"))   # short boilerplate, must not link
    df = pd.DataFrame(rows)
    for seed in range(10):
        tr, dv = group_split(df, 0.3, seed, cfg)
        side = lambda a: "dev" if a in set(dv["article_id"]) else "train"
        assert len({side(a) for a in ["a3", "a17", "a18", "a19"]}) == 1
        assert side("a25") == side("a30")
        exact, sim = train_dev_overlap(tr, dv, 0.95)
        assert not exact.any(), "identical text on both sides of the split"
    g = duplicate_aware_groups(df, cfg)
    assert len(set(g[df["article_id"].isin(["a31"]).values]) & set(g[df["article_id"].isin(["a30"]).values])) == 0


def test_crlf_offsets_and_empty_labels():
    d = os.path.join(CFG.out_dir, "_test", "crlf", "EN"); os.makedirs(os.path.join(d, "raw-documents"), exist_ok=True)
    body = "Title\r\n\r\nFirst paragraph here.\r\n\r\nSecond one.\r\n"
    s1 = body.index("First"); e1 = s1 + len("First paragraph here."); s2 = body.index("Second"); e2 = s2 + len("Second one.")
    with open(os.path.join(d, "raw-documents", "EN_X.txt"), "w", encoding="utf-8", newline="") as f:
        f.write(body)
    with open(os.path.join(d, "subtask-1-annotations.txt"), "w", encoding="utf-8", newline="") as f:
        f.write(f"EN_X.txt\t{s1}\t{e1}\tEconomic\tPolitical\r\nEN_X.txt\t{s2}\t{e2}\t\r\n")
    df = load_span_file(dict(lang="EN", origin="2027", dir=d, rel="crlf/EN",
                             path=os.path.join(d, "subtask-1-annotations.txt")), labelled=True)
    assert df["text"].tolist() == ["First paragraph here.", "Second one."], df["text"].tolist()
    assert vec_to_labels(df["labels"][0]) == ["Economic", "Political"] and df["labels"][1].sum() == 0


TESTS = [test_crlf_offsets_and_empty_labels, test_duplicate_articles_same_split, test_labels_normalise, test_loss_finite_and_matches_reference, test_export_format_roundtrip,
         test_group_split_no_leak, test_contrast_sampler_covers_classes,
         test_contrastive_loss_finite_in_training_batches, test_encoder_frozen_then_trained]
for t in TESTS:
    seed_everything(CFG.seed)
    t()
    print("PASS", t.__name__)

PASS test_crlf_offsets_and_empty_labels
PASS test_duplicate_articles_same_split
PASS test_labels_normalise
PASS test_loss_finite_and_matches_reference
PASS test_export_format_roundtrip
PASS test_group_split_no_leak
PASS test_contrast_sampler_covers_classes
PASS test_contrastive_loss_finite_in_training_batches
{'stage': 'test_head', 'epoch': 0, 'head_lr': 0.00099, 'enc_lr': 2e-05, 'bce': 0.6801}
{'stage': 'test_con', 'epoch': 0, 'head_lr': 0.0009801, 'enc_lr': 1.96e-05, 'bce': 0.6263, 'con': 0.9482}
{'stage': 'test_post', 'epoch': 0, 'head_lr': 0.00099, 'enc_lr': 1.96e-05, 'bce': 0.6775}
PASS test_encoder_frozen_then_trained


## 10. Training pipelines

**Phase 1** (pooled multilingual train): (a) head pre-training, encoder frozen, BCE, 10 ep;
(b) contrastive fine-tuning, encoder+head, BCE + α·CON, 50 ep. → checkpoint `phase1`.

**Phase 2 few-shot** (target language has labels): start from the phase-1 encoder, **re-initialise the head**,
head pre-training on target (10 ep) → contrastive fine-tuning on target → head post-training on target
(encoder frozen, BCE). Threshold tuned on the target dev split.

**Phase 2 zero-shot** (no target labels): keep the phase-1 head, post-train it on all pooled labelled languages
(encoder frozen, BCE), predict the target. Threshold tuned on pooled dev of the labelled languages.

In [ ]:
def run_phase1(cfg: Config, splits, run_dir: str, langs: Optional[Sequence[str]] = None,
               do_contrastive: bool = True, sampler: Optional[str] = None, alpha: Optional[float] = None,
               resume: Optional["MCPTTrainer"] = None) -> str:
    """resume: a partly trained phase-1 trainer (found in memory after stopping the cell). Only the epochs it has
    not done yet are run; its optimisers and LR schedules continue where they stopped."""
    seed_everything(cfg.seed)
    tr_df = pooled(splits, "train", langs, include_extra=cfg.use_semeval23_for_pretraining)
    dv_df = pooled(splits, "dev", langs)
    if resume is not None:
        tr = resume
        tr.run_dir = run_dir
        for p in tr.enc.parameters():
            p.requires_grad_(True)
    else:
        tr = MCPTTrainer(cfg, fresh_encoder(), run_dir=run_dir)
    head_done = sum(r["stage"] == "p1_head_pretrain" for r in tr.log)
    con_done = sum(r["stage"] == "p1_contrastive" for r in tr.log)
    if resume is not None:
        print(f"resuming phase 1: head {head_done}/{cfg.p1_head_epochs}, "
              f"contrastive {con_done}/{cfg.p1_con_epochs} epochs done (epoch numbers restart at 0)")
    tr.train_head(tr_df, cfg.p1_head_epochs - head_done, "p1_head_pretrain", dev_df=dv_df)
    if do_contrastive:
        tr.train_contrastive(tr_df, cfg.p1_con_epochs - con_done, "p1_contrastive", alpha=alpha, sampler=sampler,
                             dev_df=dv_df)
    res = evaluate(tr, dv_df, cfg)
    path = tr.save("phase1", extra={"pooled_dev": res})
    print("phase1 pooled dev:", res)
    if REPORT_LANG in splits and len(splits[REPORT_LANG]["dev"]):
        en = evaluate(tr, splits[REPORT_LANG]["dev"], cfg)
        print(f"phase1 {REPORT_LANG} dev: cf_micro_f1={en.get('cf_micro_f1', float('nan')):.4f} "
              f"micro_f1={en['micro_f1']:.4f} macro_f1={en['macro_f1']:.4f} threshold={en['threshold']}")
    return path


def find_stopped_phase1() -> Optional["MCPTTrainer"]:
    """A phase-1 trainer left in memory by a stopped cell (None if there is none)."""
    import gc
    found = []
    for o in gc.get_objects():
        try:
            if isinstance(o, MCPTTrainer) and any(r["stage"].startswith("p1_") for r in o.log) \
                    and not os.path.isfile(os.path.join(o.run_dir, "phase1", "checkpoint.pt")):
                found.append(o)
        except ReferenceError:
            continue
    return max(found, key=lambda o: len(o.log)) if found else None


def run_few_shot(cfg: Config, splits, target: str, run_dir: str, phase1_path: Optional[str],
                 do_contrastive: bool = True, sampler: Optional[str] = None, alpha: Optional[float] = None,
                 train_langs: Optional[Sequence[str]] = None, select: bool = False, name: Optional[str] = None,
                 save: bool = True) -> dict:
    """Phase-2 few-shot. Defaults = the paper. EN-push options:
    train_langs: 2027 training languages for phase 2 (default [target]); other languages' 2027 train is added in full.
    select: hold out an article-grouped slice of the TARGET train (cfg.select_val_frac) and keep the best epoch of
            the contrastive and post-training stages on it. The dev set is never used for selection."""
    seed_everything(cfg.seed)
    tr = MCPTTrainer(cfg, fresh_encoder(), run_dir=run_dir)
    if phase1_path is not None:   # None -> "-PT" ablation (start from the pretrained encoder)
        tr.load(phase1_path, encoder=True, head=False, enc_optim=cfg.restore_encoder_optimizer_in_p2)
    tr.reinit_head(cfg.seed)
    t_tr, t_dv = splits[target]["train"], splits[target]["dev"]
    sel_df = None
    if select:
        t_tr, sel_df = group_split(t_tr, cfg.select_val_frac, cfg.seed, cfg)
    others = [l for l in (train_langs or [target]) if l != target and l in splits]
    fit = _cat([t_tr] + [splits[l]["train"] for l in others])
    tag = name or f"p2_{target}"
    tr.train_head(fit, cfg.p2_head_epochs, f"{tag}_head_pretrain", dev_df=t_dv)
    if do_contrastive:
        tr.train_contrastive(fit, cfg.p2_con_epochs, f"{tag}_contrastive", alpha=alpha, sampler=sampler, dev_df=t_dv,
                             select_df=sel_df)
    tr.train_head(fit, cfg.p2_post_epochs, f"{tag}_head_posttrain", dev_df=t_dv, select_df=sel_df)
    res = evaluate(tr, t_dv, cfg)
    if save:
        tr.save(f"phase2_fewshot_{target}" if name is None else name, extra={"dev": res})
    print(f"few-shot {target} [{tag}] dev:", res)
    print(f"  >> {tag}: {target} cf micro-F1 = {res.get('cf_micro_f1', res['micro_f1']):.4f} "
          f"(micro {res['micro_f1']:.4f}, macro {res['macro_f1']:.4f}, threshold {res['threshold']})")
    return dict(trainer=tr, dev=res)


def run_zero_shot(cfg: Config, splits, target: str, run_dir: str, phase1_path: str,
                  source_langs: Optional[Sequence[str]] = None) -> dict:
    seed_everything(cfg.seed)
    src = [l for l in (source_langs or list(splits)) if l != target]
    tr = MCPTTrainer(cfg, fresh_encoder(), run_dir=run_dir)
    tr.load(phase1_path, encoder=True, head=True, enc_optim=False)      # keep head
    tr.train_head(pooled(splits, "train", src), cfg.zs_post_epochs, f"zs_{target}_head_posttrain",
                  dev_df=pooled(splits, "dev", src))
    src_dev = evaluate(tr, pooled(splits, "dev", src), cfg)               # threshold from source languages
    out = dict(trainer=tr, threshold=src_dev["threshold"], source_dev=src_dev)
    if target in splits and len(splits[target]["dev"]):   # simulated zero-shot: report target dev with the source-tuned threshold
        out["target_dev"] = evaluate(tr, splits[target]["dev"], cfg, threshold=src_dev["threshold"])
    tr.save(f"phase2_zeroshot_{target}", extra={k: v for k, v in out.items() if k != "trainer"})
    print(f"zero-shot {target}:", {k: v for k, v in out.items() if k != "trainer"})
    return out

In [ ]:
# === PATCH: phase 1 keeps its best epoch for EN + early stopping ===
PHASE1_SELECT_LANG = "EN"   # None = paper schedule (all epochs)
PATIENCE = 5                # stop after this many epochs without improvement on the held-out EN slice
SEL_FRAC = getattr(CFG, "select_val_frac", 0.15)


def _train_contrastive_sel(self, df, n_epochs, stage, alpha=None, sampler=None, dev_df=None, select_df=None):
    if n_epochs <= 0:
        return
    alpha = self.cfg.alpha if alpha is None else alpha
    dl = self._loader(df, sampler or self.cfg.sampler)
    best = (-1.0, None, -1)
    Ys = np.stack(select_df["labels"].values) if select_df is not None else None
    for ep in range(n_epochs):
        self.enc.train(); self.head.train()
        lb, lc = [], []
        for ids, am, y in dl:
            ids, am, y = ids.to(DEVICE), am.to(DEVICE), y.to(DEVICE)
            z = self.enc(ids, am)
            l_bce = self.bce(self.head(z), y)
            l_con = self.con(z, y) if alpha > 0 else torch.zeros((), device=DEVICE)
            loss = l_bce + alpha * l_con
            if not torch.isfinite(loss):
                raise FloatingPointError(f"non-finite loss at {stage} epoch {ep}")
            self.enc_opt.zero_grad(set_to_none=True); self.head_opt.zero_grad(set_to_none=True)
            loss.backward()
            self.enc_opt.step(); self.head_opt.step()
            lb.append(l_bce.item()); lc.append(l_con.item())
        self.enc_sched.step(); self.head_sched.step()
        sel = {}
        if select_df is not None:
            sc = tune_threshold(Ys, self.predict_proba(select_df), self.cfg)[1]["micro_f1"]
            sel = {"sel_f1": sc}
            if sc > best[0]:
                best = (sc, ({k: v.detach().cpu().clone() for k, v in self.enc.state_dict().items()},
                             {k: v.detach().cpu().clone() for k, v in self.head.state_dict().items()}), ep)
        self._record(stage, ep, bce=float(np.mean(lb)), con=float(np.mean(lc)), **sel,
                     **self._maybe_eval(stage, ep, n_epochs, dev_df))
        if select_df is not None and ep - best[2] >= PATIENCE:
            print(f"  [{stage}] early stop at epoch {ep}: no improvement for {PATIENCE} epochs")
            break
    if select_df is not None and best[1] is not None:
        self.enc.load_state_dict(best[1][0]); self.head.load_state_dict(best[1][1])
        print(f"  [{stage}] restored best epoch {best[2]} (sel_f1={best[0]:.4f})")


MCPTTrainer.train_contrastive = _train_contrastive_sel


def run_phase1(cfg, splits, run_dir, langs=None, do_contrastive=True, sampler=None, alpha=None,
               resume=None, select_lang=None):
    seed_everything(cfg.seed)
    tr_df = pooled(splits, "train", langs, include_extra=cfg.use_semeval23_for_pretraining)
    dv_df = pooled(splits, "dev", langs)
    sel_df = None
    if select_lang and select_lang in splits and len(splits[select_lang]["train"]):
        _, sel_df = group_split(splits[select_lang]["train"], SEL_FRAC, cfg.seed, cfg)   # same slice phase 2 uses
        tr_df = tr_df[~tr_df["article_id"].isin(set(sel_df["article_id"]))].reset_index(drop=True)
        print(f"phase 1: {len(sel_df)} {select_lang} train paragraphs held out for epoch selection")
    tr = resume if resume is not None else MCPTTrainer(cfg, fresh_encoder(), run_dir=run_dir)
    tr.run_dir = run_dir
    for p in tr.enc.parameters():
        p.requires_grad_(True)
    head_done = sum(r["stage"] == "p1_head_pretrain" for r in tr.log)
    con_done = sum(r["stage"] == "p1_contrastive" for r in tr.log)
    tr.train_head(tr_df, cfg.p1_head_epochs - head_done, "p1_head_pretrain", dev_df=dv_df)
    if do_contrastive:
        tr.train_contrastive(tr_df, cfg.p1_con_epochs - con_done, "p1_contrastive", alpha=alpha, sampler=sampler,
                             dev_df=dv_df, select_df=sel_df)
    res = evaluate(tr, dv_df, cfg)
    path = tr.save("phase1", extra={"pooled_dev": res})
    print("phase1 pooled dev:", res)
    if "EN" in splits and len(splits["EN"]["dev"]):
        en = evaluate(tr, splits["EN"]["dev"], cfg)
        print(f"phase1 EN dev: cf_micro_f1={en.get('cf_micro_f1', float('nan')):.4f} micro_f1={en['micro_f1']:.4f} "
              f"macro_f1={en['macro_f1']:.4f} threshold={en['threshold']}")
    return path


print("patch loaded: phase 1 selects on", PHASE1_SELECT_LANG, "| patience", PATIENCE)

patch loaded: phase 1 selects on EN | patience 5


In [ ]:
RUN_DIR = os.path.join(CFG.out_dir, "mcpt_main")
P1 = run_phase1(CFG, SPLITS, RUN_DIR, select_lang=PHASE1_SELECT_LANG)

phase 1: 199 EN train paragraphs held out for epoch selection
{'stage': 'p1_head_pretrain', 'epoch': 0, 'head_lr': 0.00099, 'enc_lr': 2e-05, 'bce': 0.2878}
{'stage': 'p1_head_pretrain', 'epoch': 1, 'head_lr': 0.0009801, 'enc_lr': 2e-05, 'bce': 0.2382}
{'stage': 'p1_head_pretrain', 'epoch': 2, 'head_lr': 0.0009703, 'enc_lr': 2e-05, 'bce': 0.2295}
{'stage': 'p1_head_pretrain', 'epoch': 3, 'head_lr': 0.0009606, 'enc_lr': 2e-05, 'bce': 0.2247}
  >> p1_head_pretrain epoch 4: EN cf micro-F1 = 0.3577
{'stage': 'p1_head_pretrain', 'epoch': 4, 'head_lr': 0.000951, 'enc_lr': 2e-05, 'bce': 0.2208, 'dev_micro_f1': 0.4438, 'dev_macro_f1': 0.373, 'dev_thr': 0.2, 'dev_cf_micro_f1': 0.4119, 'BG_cf_micro_f1': 0.4728, 'EN_cf_micro_f1': 0.3577, 'PT_cf_micro_f1': 0.4542, 'RU_cf_micro_f1': 0.4917}
{'stage': 'p1_head_pretrain', 'epoch': 5, 'head_lr': 0.0009415, 'enc_lr': 2e-05, 'bce': 0.217}
{'stage': 'p1_head_pretrain', 'epoch': 6, 'head_lr': 0.0009321, 'enc_lr': 2e-05, 'bce': 0.2147}
{'stage': 'p1_head_pr

In [ ]:
FEWSHOT = {lang: run_few_shot(CFG, SPLITS, lang, RUN_DIR, P1) for lang in TARGETS}
print(pd.DataFrame({l: r["dev"] for l, r in FEWSHOT.items()}).T.round(3))

{'stage': 'p2_EN_head_pretrain', 'epoch': 0, 'head_lr': 0.00099, 'enc_lr': 1.736e-05, 'bce': 0.4496}
{'stage': 'p2_EN_head_pretrain', 'epoch': 1, 'head_lr': 0.0009801, 'enc_lr': 1.736e-05, 'bce': 0.2827}
{'stage': 'p2_EN_head_pretrain', 'epoch': 2, 'head_lr': 0.0009703, 'enc_lr': 1.736e-05, 'bce': 0.247}
{'stage': 'p2_EN_head_pretrain', 'epoch': 3, 'head_lr': 0.0009606, 'enc_lr': 1.736e-05, 'bce': 0.223}
  >> p2_EN_head_pretrain epoch 4: EN cf micro-F1 = 0.3587
{'stage': 'p2_EN_head_pretrain', 'epoch': 4, 'head_lr': 0.000951, 'enc_lr': 1.736e-05, 'bce': 0.2066, 'dev_micro_f1': 0.3731, 'dev_macro_f1': 0.3133, 'dev_thr': 0.2, 'dev_cf_micro_f1': 0.3587}
{'stage': 'p2_EN_head_pretrain', 'epoch': 5, 'head_lr': 0.0009415, 'enc_lr': 1.736e-05, 'bce': 0.1946}
{'stage': 'p2_EN_head_pretrain', 'epoch': 6, 'head_lr': 0.0009321, 'enc_lr': 1.736e-05, 'bce': 0.185}
{'stage': 'p2_EN_head_pretrain', 'epoch': 7, 'head_lr': 0.0009227, 'enc_lr': 1.736e-05, 'bce': 0.1772}
{'stage': 'p2_EN_head_pretrain', 

KeyboardInterrupt: 

In [ ]:
FEWSHOT = {"EN": run_few_shot(CFG, SPLITS, "EN", RUN_DIR, P1, select=True, name="p2_EN_sel")}
print(pd.DataFrame({l: r["dev"] for l, r in FEWSHOT.items()}).T.round(3))

{'stage': 'p2_EN_sel_head_pretrain', 'epoch': 0, 'head_lr': 0.00099, 'enc_lr': 1.736e-05, 'bce': 0.4744}
{'stage': 'p2_EN_sel_head_pretrain', 'epoch': 1, 'head_lr': 0.0009801, 'enc_lr': 1.736e-05, 'bce': 0.2885}
{'stage': 'p2_EN_sel_head_pretrain', 'epoch': 2, 'head_lr': 0.0009703, 'enc_lr': 1.736e-05, 'bce': 0.2525}
{'stage': 'p2_EN_sel_head_pretrain', 'epoch': 3, 'head_lr': 0.0009606, 'enc_lr': 1.736e-05, 'bce': 0.2306}
  >> p2_EN_sel_head_pretrain epoch 4: EN cf micro-F1 = 0.3468
{'stage': 'p2_EN_sel_head_pretrain', 'epoch': 4, 'head_lr': 0.000951, 'enc_lr': 1.736e-05, 'bce': 0.2144, 'dev_micro_f1': 0.3612, 'dev_macro_f1': 0.2868, 'dev_thr': 0.2, 'dev_cf_micro_f1': 0.3468}
{'stage': 'p2_EN_sel_head_pretrain', 'epoch': 5, 'head_lr': 0.0009415, 'enc_lr': 1.736e-05, 'bce': 0.1995}
{'stage': 'p2_EN_sel_head_pretrain', 'epoch': 6, 'head_lr': 0.0009321, 'enc_lr': 1.736e-05, 'bce': 0.1856}
{'stage': 'p2_EN_sel_head_pretrain', 'epoch': 7, 'head_lr': 0.0009227, 'enc_lr': 1.736e-05, 'bce': 0.

In [ ]:
# === SAVE the EN model (cf micro-F1 ~0.40) to Google Drive, plus its EN leaderboard predictions ===
import json
SAVE_TO_DRIVE = True
NAME = "EN_p2_EN_sel_cf0.40"
if SAVE_TO_DRIVE:
    if not os.path.isdir("/content/drive/MyDrive"):
        from google.colab import drive
        drive.mount("/content/drive")
    SAVE_DIR = f"/content/drive/MyDrive/mcpt_models/{NAME}"
else:
    SAVE_DIR = os.path.join(CFG.out_dir, "saved_models", NAME)

m = FEWSHOT["EN"]
tr_save, res_save = m["trainer"], m["dev"]
os.makedirs(SAVE_DIR, exist_ok=True)

# 1) encoder + tokenizer (Hugging Face format), head weights, metadata
tok_to_save = TOKENIZER if globals().get("CURRENT_BACKBONE", CFG.model_name) == CFG.model_name \
    else AutoTokenizer.from_pretrained(CFG.model_name, token=False)
tr_save.enc.model.save_pretrained(os.path.join(SAVE_DIR, "encoder"))
tok_to_save.save_pretrained(os.path.join(SAVE_DIR, "encoder"))
torch.save(tr_save.head.state_dict(), os.path.join(SAVE_DIR, "head.pt"))
meta = {"name": NAME, "base_model": CFG.model_name, "threshold": res_save["threshold"], "dev_EN": res_save,
        "labels": LABELS, "max_len": MAX_LEN, "head_variant": CFG.head_variant, "head_hidden": CFG.head_hidden,
        "head_dropout": CFG.head_dropout, "force_at_least_one": CFG.force_at_least_one, "config": asdict(CFG)}
with open(os.path.join(SAVE_DIR, "meta.json"), "w") as f:
    json.dump(meta, f, indent=1, default=str)
with open(os.path.join(SAVE_DIR, "train_log.json"), "w") as f:
    json.dump(tr_save.log, f, indent=1, default=float)

# 2) EN predictions with this model: leaderboard file + labelled dev
for src in SOURCES["predict"]:
    if src["lang"] != "EN":
        continue
    df = load_span_file(src, labelled=False)
    Yh = binarize(tr_save.predict_proba(df), res_save["threshold"], CFG.force_at_least_one)
    out = os.path.join(SAVE_DIR, src["rel"].replace(os.sep, "__") + "__subtask-1-predictions.txt")
    write_submission(df, Yh, out, CFG)
    print("wrote", out, f"({len(df)} paragraphs, {int((Yh.sum(1) == 0).sum())} with no framing)")
dv = SPLITS["EN"]["dev"]
write_submission(dv, binarize(tr_save.predict_proba(dv), res_save["threshold"], CFG.force_at_least_one),
                 os.path.join(SAVE_DIR, "labelled_dev_EN_pred.txt"), CFG)

print("saved to", SAVE_DIR)
for fn in sorted(os.listdir(SAVE_DIR)):
    print("  ", fn)
print(f"EN dev cf micro-F1 = {res_save.get('cf_micro_f1', res_save['micro_f1']):.4f}, threshold = {res_save['threshold']}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

wrote /content/drive/MyDrive/mcpt_models/EN_p2_EN_sel_cf0.40/3_dev-set-nov-release__EN__subtask-1-predictions.txt (558 paragraphs, 151 with no framing)
saved to /content/drive/MyDrive/mcpt_models/EN_p2_EN_sel_cf0.40
   3_dev-set-nov-release__EN__subtask-1-predictions.txt
   encoder
   head.pt
   labelled_dev_EN_pred.txt
   meta.json
   train_log.json
EN dev cf micro-F1 = 0.3993, threshold = 0.35


In [ ]:
SAVE_DIR = "/content/drive/MyDrive/mcpt_models/EN_p2_EN_sel_cf0.40"

def load_saved_model(path):
    meta = json.load(open(os.path.join(path, "meta.json")))
    global TOKENIZER, MAX_LEN
    TOKENIZER = AutoTokenizer.from_pretrained(os.path.join(path, "encoder"))
    MAX_LEN = meta["max_len"]
    enc = Encoder(AutoModel.from_pretrained(os.path.join(path, "encoder")))
    cfg = dataclasses.replace(CFG, head_variant=meta["head_variant"], head_hidden=meta["head_hidden"],
                              head_dropout=meta["head_dropout"])
    t = MCPTTrainer(cfg, enc, run_dir=os.path.join(CFG.out_dir, "loaded"))
    t.head.load_state_dict(torch.load(os.path.join(path, "head.pt"), map_location=DEVICE))
    return t, meta["threshold"], meta

tr_loaded, thr_loaded, meta_loaded = load_saved_model(SAVE_DIR)
print("loaded", meta_loaded["name"], "| threshold", thr_loaded)
print(evaluate(tr_loaded, SPLITS["EN"]["dev"], CFG, threshold=thr_loaded))   # reproduces the saved dev scores

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

loaded EN_p2_EN_sel_cf0.40 | threshold 0.35
{'threshold': 0.35, 'micro_f1': 0.39933444259567386, 'macro_f1': 0.3200229972163355, 'micro_f1_14': 0.3672566371681416, 'macro_f1_14': 0.3074071902102156}


In [ ]:
# Zero-shot: for a language with no labels, phase 1 must be trained without it. Here we simulate it for one
# labelled language (excluded from phase 1) so it can be evaluated. Set ZS_TARGET=None to skip.
ZS_TARGET = TARGETS[-1] if (SMOKE_TEST and not EN_ONLY) else None
if ZS_TARGET:
    src = [l for l in SPLITS if l != ZS_TARGET]
    P1_wo = run_phase1(CFG, SPLITS, os.path.join(CFG.out_dir, f"mcpt_wo_{ZS_TARGET}"), langs=src)
    ZS = run_zero_shot(CFG, SPLITS, ZS_TARGET, os.path.join(CFG.out_dir, f"mcpt_wo_{ZS_TARGET}"), P1_wo, src)

## 10b. EN push (goal: EN micro-F1 ≥ 0.47)
Beyond the replication, aimed at one language. Each step is measured on the EN dev set (gold 2b paragraphs) with
the **cross-fitted** micro-F1 (`cf_micro_f1`, threshold tuned on half the dev articles, scored on the other half).
Epoch selection uses a slice of EN *train*, never dev.

| id | change vs replication | why |
|---|---|---|
| E0 | mCPT replication (phase 1 + EN-only few-shot) | baseline |
| E1 | + phase-2 trained on **all 2027 train languages** (BG, EN, PT, RU) + best-epoch selection | EN has only ~1k paragraphs; same annotation scheme and NONE rate in the other 2027 batches |
| E2 | E1 with `paraphrase-multilingual-mpnet-base-v2`, no phase 1 | larger encoder; the paper's -PT ablation shows phase 1 barely helps EN (.681 vs .682) |
| E3 | E2 + phase 1 (optional, about 2 h on a T4) | `RUN_MPNET_PHASE1` |
| ENS | 3-seed average of the best of the above | variance reduction; threshold re-tuned on averaged probabilities |

Measured on the real data before any of this: a TF-IDF + logistic regression baseline reaches cf micro-F1 ≈ 0.36
on EN dev (majority-label baseline 0.24). Per-label thresholds and adding the previous paragraph as context
lowered it, so they are not used here.

In [ ]:
# === KEEP + SAVE THE HIGHEST-SCORING EN MODEL ===
import json
CFG.eval_every = 1                      # evaluate EN every epoch, so no good epoch is missed
BEST_EN_DIR = "/content/drive/MyDrive/mcpt_models/EN_best_on_dev"
if BEST_EN_DIR.startswith("/content/drive") and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")
BEST_EN = {"score": -1.0}
_BEST_DIRTY = [False]
_orig_maybe_eval = MCPTTrainer._maybe_eval


def _snapshot(trainer, score, info):
    BEST_EN.update(
        score=score, info=info, threshold=info.get("dev_thr", info.get("threshold")),
        enc_state={k: v.detach().cpu().clone() for k, v in trainer.enc.model.state_dict().items()},
        head_state={k: v.detach().cpu().clone() for k, v in trainer.head.state_dict().items()},
        base=_ENC0, tokenizer=TOKENIZER, max_len=MAX_LEN, backbone=globals().get("CURRENT_BACKBONE", CFG.model_name),
        dim=trainer.enc.dim)
    _BEST_DIRTY[0] = True
    print(f"  ** new best EN model: cf micro-F1 = {score:.4f} ({info.get('where')})")


def _maybe_eval_keep(self, stage, epoch, n_epochs, dev_df):
    out = _orig_maybe_eval(self, stage, epoch, n_epochs, dev_df)
    if out and dev_df is not None and len(dev_df):
        langs = set(dev_df["lang"])
        score = out.get("EN_cf_micro_f1", out.get("dev_cf_micro_f1") if langs == {"EN"} else None)
        if score is not None and score > BEST_EN["score"]:
            dev_thr = out["dev_thr"]
            if langs != {"EN"}:   # pooled dev: re-tune the threshold on EN only
                en = SPLITS["EN"]["dev"]
                dev_thr = tune_threshold(np.stack(en["labels"].values), self.predict_proba(en), self.cfg)[0]
            _snapshot(self, score, {"where": f"{stage} epoch {epoch}", "dev_thr": dev_thr, **out})
    return out


MCPTTrainer._maybe_eval = _maybe_eval_keep


@torch.no_grad()
def _predict_with_snapshot(enc, head, tok, max_len, df, bs=128):
    enc.eval(); head.eval()
    P = []
    texts = df["text"].tolist()
    for i in range(0, len(texts), bs):
        b = tok(texts[i:i + bs], truncation=True, max_length=max_len, padding=True, return_tensors="pt")
        z = enc(b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE))
        P.append(torch.sigmoid(head(z)).cpu())
    return torch.cat(P).numpy() if P else np.zeros((0, NUM_CLASSES))


def save_best_en(path=None):
    if BEST_EN["score"] < 0:
        print("no EN model evaluated yet"); return None
    path = path or BEST_EN_DIR
    os.makedirs(path, exist_ok=True)
    hf = copy.deepcopy(BEST_EN["base"]); hf.load_state_dict(BEST_EN["enc_state"])
    enc = Encoder(hf).to(DEVICE)
    head = build_head(CFG, BEST_EN["dim"]); head.load_state_dict(BEST_EN["head_state"]); head.to(DEVICE)
    hf.save_pretrained(os.path.join(path, "encoder"))
    BEST_EN["tokenizer"].save_pretrained(os.path.join(path, "encoder"))
    torch.save(BEST_EN["head_state"], os.path.join(path, "head.pt"))
    meta = {"name": "EN_best_on_dev", "base_model": BEST_EN["backbone"], "threshold": BEST_EN["threshold"],
            "en_cf_micro_f1": BEST_EN["score"], "selected_at": BEST_EN["info"].get("where"),
            "eval_line": {k: v for k, v in BEST_EN["info"].items() if isinstance(v, (int, float, str))},
            "note": "selected as the max over many EN dev evaluations: its dev score is optimistic",
            "labels": LABELS, "max_len": BEST_EN["max_len"], "head_variant": CFG.head_variant,
            "head_hidden": CFG.head_hidden, "head_dropout": CFG.head_dropout, "config": asdict(CFG)}
    with open(os.path.join(path, "meta.json"), "w") as f:
        json.dump(meta, f, indent=1, default=str)
    thr = BEST_EN["threshold"]
    for src in SOURCES["predict"]:
        if src["lang"] != "EN":
            continue
        df = load_span_file(src, labelled=False)
        Yh = binarize(_predict_with_snapshot(enc, head, BEST_EN["tokenizer"], BEST_EN["max_len"], df), thr,
                      CFG.force_at_least_one)
        write_submission(df, Yh, os.path.join(path, src["rel"].replace(os.sep, "__") + "__subtask-1-predictions.txt"), CFG)
    dv = SPLITS["EN"]["dev"]
    P = _predict_with_snapshot(enc, head, BEST_EN["tokenizer"], BEST_EN["max_len"], dv)
    chk = scores(np.stack(dv["labels"].values), binarize(P, thr, CFG.force_at_least_one))
    del enc, head, hf
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    _BEST_DIRTY[0] = False
    print(f"saved best EN model -> {path} | cf micro-F1 {BEST_EN['score']:.4f} at {BEST_EN['info'].get('where')} "
          f"| threshold {thr} | check: dev micro-F1 at that threshold = {chk['micro_f1']:.4f}")
    return path


_orig_run_few_shot = run_few_shot


def run_few_shot(*args, **kwargs):          # same function; best EN model is saved after every run
    r = _orig_run_few_shot(*args, **kwargs)
    target = kwargs.get("target", args[2] if len(args) > 2 else None)
    d = r["dev"]
    if target == "EN" and d.get("cf_micro_f1", -1) > BEST_EN["score"]:
        _snapshot(r["trainer"], d["cf_micro_f1"], {"where": f"end of {kwargs.get('name') or 'run'}",
                                                   "dev_thr": d["threshold"], **d})
    if _BEST_DIRTY[0]:
        save_best_en()
    return r


# start from the model you already have (EN-only, ~0.40)
if "FEWSHOT" in globals() and "EN" in FEWSHOT and "trainer" in FEWSHOT["EN"]:
    _d = FEWSHOT["EN"]["dev"]
    _snapshot(FEWSHOT["EN"]["trainer"], _d.get("cf_micro_f1", _d["micro_f1"]),
              {"where": "existing FEWSHOT['EN']", "dev_thr": _d["threshold"], **_d})
    save_best_en()
print("EN best-model keeper active ->", BEST_EN_DIR)

  ** new best EN model: cf micro-F1 = 0.3993 (existing FEWSHOT['EN'])


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved best EN model -> /content/drive/MyDrive/mcpt_models/EN_best_on_dev | cf micro-F1 0.3993 at existing FEWSHOT['EN'] | threshold 0.35 | check: dev micro-F1 at that threshold = 0.3993
EN best-model keeper active -> /content/drive/MyDrive/mcpt_models/EN_best_on_dev


In [ ]:
RUN_EN_PUSH = True
EN_TARGET, EN_GOAL = "EN", 0.47
MPNET = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
RUN_MPNET_PHASE1 = False
N_SEEDS = 3
ALL27 = list(LANGS_2027)


def _probs_for(trainer, dfs: Dict[str, pd.DataFrame]) -> Dict[str, np.ndarray]:
    return {k: trainer.predict_proba(v) for k, v in dfs.items()}


def _free(res):
    del res["trainer"]
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


if RUN_EN_PUSH and EN_TARGET in SPLITS and len(SPLITS[EN_TARGET]["dev"]):
    en_dev = SPLITS[EN_TARGET]["dev"]
    en_pred = {s["rel"]: load_span_file(s, False) for s in SOURCES["predict"] if s["lang"] == EN_TARGET}
    EXPS = {}            # name -> dict(dev=metrics, spec=how to rerun)
    EXPS["E0_replication"] = dict(dev=FEWSHOT[EN_TARGET]["dev"], spec=None)

    use_backbone(CFG.model_name)
    r = run_few_shot(CFG, SPLITS, EN_TARGET, os.path.join(CFG.out_dir, "en_push"), P1,
                     train_langs=ALL27, select=True, name="E1_minilm_p1_multi_sel")
    EXPS["E1_minilm_p1_multi_sel"] = dict(dev=r["dev"], spec=dict(backbone=CFG.model_name, p1=P1)); _free(r)

    use_backbone(MPNET)
    r = run_few_shot(CFG, SPLITS, EN_TARGET, os.path.join(CFG.out_dir, "en_push"), None,
                     train_langs=ALL27, select=True, name="E2_mpnet_multi_sel")
    EXPS["E2_mpnet_multi_sel"] = dict(dev=r["dev"], spec=dict(backbone=MPNET, p1=None)); _free(r)

    if RUN_MPNET_PHASE1:
        P1_MPNET = run_phase1(CFG, SPLITS, os.path.join(CFG.out_dir, "en_push_mpnet"))
        r = run_few_shot(CFG, SPLITS, EN_TARGET, os.path.join(CFG.out_dir, "en_push_mpnet"), P1_MPNET,
                         train_langs=ALL27, select=True, name="E3_mpnet_p1_multi_sel")
        EXPS["E3_mpnet_p1_multi_sel"] = dict(dev=r["dev"], spec=dict(backbone=MPNET, p1=P1_MPNET)); _free(r)

    table = pd.DataFrame({k: v["dev"] for k, v in EXPS.items()}).T
    print(table[["cf_micro_f1", "micro_f1", "macro_f1", "cf_macro_f1", "threshold"]].round(3))

    # ---- seed ensemble of the best runnable config (by cross-fitted micro-F1) ----
    runnable = {k: v for k, v in EXPS.items() if v["spec"] is not None}
    best_name = max(runnable, key=lambda k: runnable[k]["dev"].get("cf_micro_f1", runnable[k]["dev"]["micro_f1"]))
    spec = runnable[best_name]["spec"]
    print("ensembling", best_name, "over", N_SEEDS, "seeds")
    use_backbone(spec["backbone"])
    Pdev, Ppred = [], {k: [] for k in en_pred}
    for i in range(N_SEEDS):
        cfg_i = dataclasses.replace(CFG, seed=CFG.seed + i)
        r = run_few_shot(cfg_i, SPLITS, EN_TARGET, os.path.join(CFG.out_dir, "en_push_ens"), spec["p1"],
                         train_langs=ALL27, select=True, name=f"ENS_{best_name}_seed{cfg_i.seed}", save=False)
        Pdev.append(r["trainer"].predict_proba(en_dev))
        for k, v in en_pred.items():
            Ppred[k].append(r["trainer"].predict_proba(v))
        _free(r)
    Pdev = np.mean(Pdev, 0)
    Ydev = np.stack(en_dev["labels"].values)
    thr, ens = tune_threshold(Ydev, Pdev, CFG)
    ens.update(crossfit_scores(Ydev, Pdev, en_dev["article_id"].astype(str).values, CFG))
    EXPS[f"ENS{N_SEEDS}_{best_name}"] = dict(dev=dict(threshold=thr, **ens), spec=None)
    table = pd.DataFrame({k: v["dev"] for k, v in EXPS.items()}).T
    print(table[["cf_micro_f1", "micro_f1", "macro_f1", "cf_macro_f1", "threshold"]].round(3))
    table.to_csv(os.path.join(CFG.out_dir, "en_push_results.csv"))
    hit = ens.get("cf_micro_f1", ens["micro_f1"]) >= EN_GOAL
    print(f"EN ensemble cf micro-F1 = {ens.get('cf_micro_f1', ens['micro_f1']):.3f} -> goal {EN_GOAL} "
          + ("REACHED" if hit else "not reached"))
    for k, v in en_pred.items():
        Yh = binarize(np.mean(Ppred[k], 0), thr, CFG.force_at_least_one)
        path = os.path.join(CFG.out_dir, "submissions", k.replace(os.sep, "__") + "__subtask-1-predictions_ENSEMBLE.txt")
        write_submission(v, Yh, path, CFG)
        print("wrote", path, f"({int((Yh.sum(1) == 0).sum())} paragraphs with no framing)")

    # ---- final refit on EN train + EN labelled dev ("full data") ----
    if EN_ONLY and FINAL_REFIT_WITH_DEV:
        print("FINAL REFIT on EN train + EN dev; its dev score below is in-sample and NOT an estimate")
        full = {l: dict(d) for l, d in SPLITS.items()}
        full[EN_TARGET]["train"] = _cat([SPLITS[EN_TARGET]["train"], SPLITS[EN_TARGET]["dev"]])
        Pfull = {k: [] for k in en_pred}
        for i in range(N_SEEDS):
            cfg_i = dataclasses.replace(CFG, seed=CFG.seed + i)
            r = run_few_shot(cfg_i, full, EN_TARGET, os.path.join(CFG.out_dir, "en_fullfit"), spec["p1"],
                             train_langs=ALL27, select=True, name=f"FULLFIT_{best_name}_seed{cfg_i.seed}", save=False)
            for k, v in en_pred.items():
                Pfull[k].append(r["trainer"].predict_proba(v))
            _free(r)
        for k, v in en_pred.items():
            Yh = binarize(np.mean(Pfull[k], 0), thr, CFG.force_at_least_one)   # threshold from the honest ensemble
            path = os.path.join(CFG.out_dir, "submissions",
                                k.replace(os.sep, "__") + "__subtask-1-predictions_ENSEMBLE_FULLFIT.txt")
            write_submission(v, Yh, path, CFG)
            print("wrote", path, f"({int((Yh.sum(1) == 0).sum())} paragraphs with no framing)")
    use_backbone(CFG.model_name)

  >> E1_minilm_p1_multi_sel_head_pretrain epoch 0: EN cf micro-F1 = 0.3097
{'stage': 'E1_minilm_p1_multi_sel_head_pretrain', 'epoch': 0, 'head_lr': 0.00099, 'enc_lr': 1.736e-05, 'bce': 0.3305, 'dev_micro_f1': 0.3171, 'dev_macro_f1': 0.2237, 'dev_thr': 0.2, 'dev_cf_micro_f1': 0.3097}
  >> E1_minilm_p1_multi_sel_head_pretrain epoch 1: EN cf micro-F1 = 0.4059
  ** new best EN model: cf micro-F1 = 0.4059 (E1_minilm_p1_multi_sel_head_pretrain epoch 1)
{'stage': 'E1_minilm_p1_multi_sel_head_pretrain', 'epoch': 1, 'head_lr': 0.0009801, 'enc_lr': 1.736e-05, 'bce': 0.2119, 'dev_micro_f1': 0.4059, 'dev_macro_f1': 0.3044, 'dev_thr': 0.35, 'dev_cf_micro_f1': 0.4059}
  >> E1_minilm_p1_multi_sel_head_pretrain epoch 2: EN cf micro-F1 = 0.4163
  ** new best EN model: cf micro-F1 = 0.4163 (E1_minilm_p1_multi_sel_head_pretrain epoch 2)
{'stage': 'E1_minilm_p1_multi_sel_head_pretrain', 'epoch': 2, 'head_lr': 0.0009703, 'enc_lr': 1.736e-05, 'bce': 0.1868, 'dev_micro_f1': 0.4163, 'dev_macro_f1': 0.3617, 'd

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved best EN model -> /content/drive/MyDrive/mcpt_models/EN_best_on_dev | cf micro-F1 0.4323 at E1_minilm_p1_multi_sel_head_pretrain epoch 6 | threshold 0.35 | check: dev micro-F1 at that threshold = 0.4401


config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/402 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

... (6 outputs left out)

backbone -> sentence-transformers/paraphrase-multilingual-mpnet-base-v2 | dim 768 | max_len 512
  >> E2_mpnet_multi_sel_head_pretrain epoch 0: EN cf micro-F1 = 0.2556
{'stage': 'E2_mpnet_multi_sel_head_pretrain', 'epoch': 0, 'head_lr': 0.00099, 'enc_lr': 2e-05, 'bce': 0.3344, 'dev_micro_f1': 0.2833, 'dev_macro_f1': 0.1913, 'dev_thr': 0.2, 'dev_cf_micro_f1': 0.2556}
  >> E2_mpnet_multi_sel_head_pretrain epoch 1: EN cf micro-F1 = 0.3612
{'stage': 'E2_mpnet_multi_sel_head_pretrain', 'epoch': 1, 'head_lr': 0.0009801, 'enc_lr': 2e-05, 'bce': 0.2468, 'dev_micro_f1': 0.374, 'dev_macro_f1': 0.2544, 'dev_thr': 0.3, 'dev_cf_micro_f1': 0.3612}
  >> E2_mpnet_multi_sel_head_pretrain epoch 2: EN cf micro-F1 = 0.4052
{'stage': 'E2_mpnet_multi_sel_head_pretrain', 'epoch': 2, 'head_lr': 0.0009703, 'enc_lr': 2e-05, 'bce': 0.2257, 'dev_micro_f1': 0.4052, 'dev_macro_f1': 0.3008, 'dev_thr': 0.3, 'dev_cf_micro_f1': 0.4052}
  >> E2_mpnet_multi_sel_head_pretrain epoch 3: EN cf micro-F1 = 0.4160
{'stage': 'E2_

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved best EN model -> /content/drive/MyDrive/mcpt_models/EN_best_on_dev | cf micro-F1 0.4611 at E2_mpnet_multi_sel_head_posttrain epoch 6 | threshold 0.3 | check: dev micro-F1 at that threshold = 0.4676
                        cf_micro_f1  micro_f1  macro_f1  cf_macro_f1  \
E0_replication                0.399     0.399     0.320        0.320   
E1_minilm_p1_multi_sel        0.380     0.408     0.356        0.334   
E2_mpnet_multi_sel            0.453     0.454     0.380        0.376   

                        threshold  
E0_replication               0.35  
E1_minilm_p1_multi_sel       0.30  
E2_mpnet_multi_sel           0.35  
ensembling E2_mpnet_multi_sel over 3 seeds
  >> ENS_E2_mpnet_multi_sel_seed42_head_pretrain epoch 0: EN cf micro-F1 = 0.2556
{'stage': 'ENS_E2_mpnet_multi_sel_seed42_head_pretrain', 'epoch': 0, 'head_lr': 0.00099, 'enc_lr': 2e-05, 'bce': 0.3344, 'dev_micro_f1': 0.2833, 'dev_macro_f1': 0.1913, 'dev_thr': 0.2, 'dev_cf_micro_f1': 0.2556}
  >> ENS_E2_mpnet_multi_se

KeyboardInterrupt: 

In [ ]:
# 4) anything still alive: move to CPU and drop optimizer state (skips half-built objects)
import gc, warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    _objs = gc.get_objects()
_n_moved = 0
for _o in _objs:
    try:
        if type(_o) is MCPTTrainer:
            for _attr in ("enc", "head"):
                if hasattr(_o, _attr):
                    getattr(_o, _attr).cpu()
            for _attr in ("enc_opt", "head_opt"):
                if hasattr(_o, _attr):
                    getattr(_o, _attr).state.clear()
            _n_moved += 1
    except ReferenceError:
        continue
del _objs
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"moved {_n_moved} leftover model(s) off the GPU | GPU memory in use: "
          f"{torch.cuda.memory_allocated() / 1e9:.2f} GB allocated, {torch.cuda.memory_reserved() / 1e9:.2f} GB reserved")

moved 2 leftover model(s) off the GPU | GPU memory in use: 6.32 GB allocated, 7.84 GB reserved


## 11. Ablations (paper Table 2)
| name | change |
|---|---|
| `mCPT` | full pipeline (phase 1 + few-shot phase 2) |
| `-PT` | no multilingual phase 1: phase 2 starts from the pretrained encoder |
| `-L_CON` | α = 0 in every contrastive stage (still end-to-end BCE) |
| `-E2E` | no encoder fine-tuning anywhere: only head stages (paper: "only a trained classification head with no embedding fine-tuning") |
| `mCPT+CS` | full pipeline with the contrast sampler |

Expensive: each variant reruns phase 1. Paper reference (dev micro-F1, 2023 article data):
mCPT en .682 it .585 ru .520 fr .570 ge .561 po .636.

In [ ]:
# === EN ABLATIONS ON TOP OF THE BEST EN SETUP ===
# Each variant removes ONE component from the best setup found by the EN push; the best run itself is reused
# (not retrained). If the best-model keeper cell was run, any variant that beats the best EN model is saved too.
RUN_EN_ABLATIONS = True
ABL_TARGET = "EN"

if RUN_EN_ABLATIONS and ABL_TARGET in SPLITS and len(SPLITS[ABL_TARGET]["dev"]):
    # ---- the best setup (from the EN push table), or the default E1 setup if the push has not run ----
    if "EXPS" in globals() and any(v.get("spec") for v in EXPS.values()):
        _runnable = {k: v for k, v in EXPS.items() if v.get("spec") is not None}
        BEST_SETUP = max(_runnable, key=lambda k: _runnable[k]["dev"].get("cf_micro_f1", _runnable[k]["dev"]["micro_f1"]))
        BEST_SPEC, BEST_DEV = _runnable[BEST_SETUP]["spec"], _runnable[BEST_SETUP]["dev"]
    else:
        BEST_SETUP, BEST_SPEC, BEST_DEV = "default (MiniLM + phase 1 + all 2027 langs + selection)", \
            dict(backbone=CFG.model_name, p1=P1), None
    _langs_all = ALL27 if "ALL27" in globals() else LANGS_2027
    print(f"best setup: {BEST_SETUP} | backbone {BEST_SPEC['backbone']} | phase 1 {'yes' if BEST_SPEC['p1'] else 'no'}")

    base_kw = dict(train_langs=_langs_all, select=True)
    variants = {
        "-L_CON (alpha=0 in phase 2)": dict(alpha=0.0),
        "-E2E (no encoder fine-tuning in phase 2)": dict(do_contrastive=False),
        "+CS (contrast sampler)": dict(sampler="contrast"),
        "-multi (EN data only in phase 2)": dict(train_langs=None),
    }
    # phase-1 ablation: only possible where a matching phase-1 checkpoint exists (P1 is MiniLM)
    if BEST_SPEC["p1"] is not None:
        variants["-PT (no phase 1)"] = dict(p1=None)
    elif BEST_SPEC["backbone"] == CFG.model_name and "P1" in globals():
        variants["+PT (add phase 1)"] = dict(p1=P1)
    else:
        print("[info] no phase-1 checkpoint for", BEST_SPEC["backbone"], "-> PT ablation skipped "
              "(set RUN_MPNET_PHASE1=True in the EN push to get one)")

    use_backbone(BEST_SPEC["backbone"])
    ABL_EN = {}
    if BEST_DEV is not None:
        ABL_EN[f"BEST: {BEST_SETUP} (reused)"] = BEST_DEV
    else:
        _r = run_few_shot(CFG, SPLITS, ABL_TARGET, os.path.join(CFG.out_dir, "abl_en"), BEST_SPEC["p1"],
                          name="ABL_best", **base_kw)
        ABL_EN["BEST: default"] = _r["dev"]; del _r["trainer"]
    for vname, change in variants.items():
        kw = dict(base_kw); kw.update({k: v for k, v in change.items() if k != "p1"})
        p1 = change.get("p1", BEST_SPEC["p1"])
        tag = "ABL_" + re.sub(r"\W+", "_", vname.split(" (")[0]).strip("_")
        _r = run_few_shot(CFG, SPLITS, ABL_TARGET, os.path.join(CFG.out_dir, "abl_en"), p1, name=tag, **kw)
        ABL_EN[vname] = _r["dev"]
        del _r["trainer"]
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    use_backbone(CFG.model_name)

    abl = pd.DataFrame(ABL_EN).T[["cf_micro_f1", "micro_f1", "cf_macro_f1", "macro_f1", "threshold"]]
    ref = abl.iloc[0]["cf_micro_f1"]
    abl.insert(1, "delta_vs_best", abl["cf_micro_f1"] - ref)
    print(f"\nEN ablations (cf = cross-fitted, honest; delta = change in EN cf micro-F1 vs the best setup)")
    print(abl.round(3).to_string())
    abl.to_csv(os.path.join(CFG.out_dir, "ablation_EN.csv"))
    if "BEST_EN" in globals():
        print(f"\nbest EN model saved so far: cf micro-F1 {BEST_EN['score']:.4f} ({BEST_EN['info'].get('where')})")

best setup: E2_mpnet_multi_sel | backbone sentence-transformers/paraphrase-multilingual-mpnet-base-v2 | phase 1 no
[info] no phase-1 checkpoint for sentence-transformers/paraphrase-multilingual-mpnet-base-v2 -> PT ablation skipped (set RUN_MPNET_PHASE1=True in the EN push to get one)
  >> ABL_L_CON_head_pretrain epoch 0: EN cf micro-F1 = 0.2556
{'stage': 'ABL_L_CON_head_pretrain', 'epoch': 0, 'head_lr': 0.00099, 'enc_lr': 2e-05, 'bce': 0.3344, 'dev_micro_f1': 0.2833, 'dev_macro_f1': 0.1913, 'dev_thr': 0.2, 'dev_cf_micro_f1': 0.2556}
  >> ABL_L_CON_head_pretrain epoch 1: EN cf micro-F1 = 0.3612
{'stage': 'ABL_L_CON_head_pretrain', 'epoch': 1, 'head_lr': 0.0009801, 'enc_lr': 2e-05, 'bce': 0.2468, 'dev_micro_f1': 0.374, 'dev_macro_f1': 0.2544, 'dev_thr': 0.3, 'dev_cf_micro_f1': 0.3612}
  >> ABL_L_CON_head_pretrain epoch 2: EN cf micro-F1 = 0.4052
{'stage': 'ABL_L_CON_head_pretrain', 'epoch': 2, 'head_lr': 0.0009703, 'enc_lr': 2e-05, 'bce': 0.2257, 'dev_micro_f1': 0.4052, 'dev_macro_f1': 0

KeyboardInterrupt: 

## 12. Predict & export submission files
For each language: load the unlabelled template (`{split}-template-subtask-1.txt`: `file \t start \t end`),
predict with that language's few-shot model and its dev-tuned threshold, write the official format.
The threshold always comes from dev; nothing is tuned on test.

In [ ]:
def predict_units(cfg: Config, trainer: MCPTTrainer, threshold: float, df: pd.DataFrame, name: str) -> str:
    Yh = binarize(trainer.predict_proba(df), threshold, cfg.force_at_least_one)
    path = os.path.join(cfg.out_dir, "submissions", f"{name}.txt")
    write_submission(df, Yh, path, cfg)
    print(f"wrote {path}: {len(df)} units, {int((Yh.sum(1) == 0).sum())} with no framing")
    return path


# every subtask-1-participant.txt in the release (dev leaderboard now, test later) -> one file per language,
# predicted by that language's few-shot model with its dev-tuned threshold
for src in SOURCES["predict"]:
    lang = src["lang"]
    if lang in FEWSHOT:
        tr_, thr_ = FEWSHOT[lang]["trainer"], FEWSHOT[lang]["dev"]["threshold"]
    elif ZS_TARGET == lang:
        tr_, thr_ = ZS["trainer"], ZS["threshold"]
    else:
        print(f"[skip] no model for {lang}"); continue
    predict_units(CFG, tr_, thr_, load_span_file(src, labelled=False),
                  src["rel"].replace(os.sep, "__") + "__subtask-1-predictions")

In [ ]:
# Dev gold vs prediction files (only possible where dev labels exist, i.e. the internal split), for the
# official scorer if you have it.
for lang, r in FEWSHOT.items():
    dv = SPLITS[lang]["dev"]
    gold_p = os.path.join(CFG.out_dir, "submissions", f"labelled_dev_gold_{lang}.txt")
    pred_p = os.path.join(CFG.out_dir, "submissions", f"labelled_dev_pred_{lang}.txt")
    write_submission(dv, np.stack(dv["labels"].values), gold_p, CFG)
    write_submission(dv, binarize(r["trainer"].predict_proba(dv), r["dev"]["threshold"], CFG.force_at_least_one), pred_p, CFG)
    run_official_scorer(gold_p, pred_p, CFG)

with open(os.path.join(CFG.out_dir, "summary.json"), "w") as f:
    json.dump({"config": asdict(CFG), "fewshot_dev": {l: r["dev"] for l, r in FEWSHOT.items()}},
              f, indent=1, default=str)
print("done ->", CFG.out_dir)

In [ ]:
# === VALIDATE + DOWNLOAD the EN submission from the best saved model ===
import glob, json, shutil
BEST_DIR = "/content/drive/MyDrive/mcpt_models/EN_best_on_dev"
meta = json.load(open(os.path.join(BEST_DIR, "meta.json")))
print(f"model: {meta['base_model']} | selected at {meta['selected_at']} | EN cf micro-F1 {meta['en_cf_micro_f1']:.4f} "
      f"| threshold {meta['threshold']}")

src = [s for s in SOURCES["predict"] if s["lang"] == "EN"][0]           # official EN participant file
pred_path = os.path.join(BEST_DIR, src["rel"].replace(os.sep, "__") + "__subtask-1-predictions.txt")
assert os.path.isfile(pred_path), f"missing {pred_path}: run save_best_en() first"

# ---- format checks against the participant file ----
ref = _parse_span_file(src["path"], with_labels=False)                # (file, start, end, [])
lines = open(pred_path, encoding="utf-8").read().splitlines()
pred = [l.split("\t") for l in lines if l.strip()]
problems = []
if len(pred) != len(ref):
    problems.append(f"{len(pred)} prediction lines vs {len(ref)} paragraphs in the participant file")
for i, (p, r) in enumerate(zip(pred, ref)):
    if len(p) < 3 or (p[0], int(p[1]), int(p[2])) != (r[0], r[1], r[2]):
        problems.append(f"line {i + 1}: {p[:3]} does not match participant {r[:3]}"); break
    bad = [x for x in p[3:] if x and x not in LABELS]
    if bad:
        problems.append(f"line {i + 1}: unknown labels {bad}"); break
n_empty = sum(1 for p in pred if not any(x for x in p[3:]))
lab_counts = pd.Series([x for p in pred for x in p[3:] if x]).value_counts()
print(f"{len(pred)} paragraphs | {n_empty} with no framing ({n_empty / max(1, len(pred)):.0%}) | "
      f"{len(lab_counts)} distinct labels used")
print(lab_counts.to_string())
print("FORMAT OK" if not problems else "PROBLEMS:\n  " + "\n  ".join(problems))

# ---- copy with a clean name and download ----
final = "/content/EN_subtask-1-predictions.txt"
shutil.copy(pred_path, final)
print("\nfirst lines:"); print("\n".join(lines[:5]))
try:
    from google.colab import files
    files.download(final)
except Exception as e:
    print("download manually from", final, "|", e)

model: sentence-transformers/paraphrase-multilingual-mpnet-base-v2 | selected at E2_mpnet_multi_sel_head_posttrain epoch 6 | EN cf micro-F1 0.4611 | threshold 0.3
558 paragraphs | 82 with no framing (15%) | 14 distinct labels used
Security_and_defense                            166
Political                                       142
Crime_and_punishment                            114
Policy_prescription_and_evaluation              100
Legality_Constitutionality_and_jurisprudence     79
Quality_of_life                                  67
Capacity_and_resources                           55
Health_and_safety                                54
External_regulation_and_reputation               30
Economic                                         25
Public_opinion                                   24
Fairness_and_equality                            13
Morality                                         12
Cultural_identity                                 3
FORMAT OK

first lines:
EN_M_TRAIN_101.tx

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# === BUILD THE CODABENCH ZIP (prediction file at the zip root) ===
import zipfile, shutil, os
PRED_TXT = "/content/EN_subtask-1-predictions.txt"           # created by the validation cell
INNER_NAME = "subtask_1_best_prediction.txt"                 # file name inside the zip (root, no folders)
ZIP_OUT = "/content/EN_subtask1_submission.zip"

assert os.path.isfile(PRED_TXT) and os.path.getsize(PRED_TXT) > 0, "prediction file missing or empty"
if os.path.exists(ZIP_OUT):
    os.remove(ZIP_OUT)
with zipfile.ZipFile(ZIP_OUT, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_TXT, arcname=INNER_NAME)                    # arcname without a path -> zip root

# ---- check it like the upload page does ----
with zipfile.ZipFile(ZIP_OUT) as z:
    names = z.namelist()
    assert z.testzip() is None, "zip is corrupted"
    assert names == [INNER_NAME], f"zip must contain exactly one file at the root, has {names}"
    n_lines = len(z.read(INNER_NAME).decode("utf-8").splitlines())
print(f"{ZIP_OUT}: {os.path.getsize(ZIP_OUT):,} bytes | contains {names} | {n_lines} lines | OK")

shutil.copy(ZIP_OUT, "/content/drive/MyDrive/mcpt_models/EN_best_on_dev/") if os.path.isdir(
    "/content/drive/MyDrive/mcpt_models/EN_best_on_dev") else None   # keep a copy in Drive
try:
    from google.colab import files
    files.download(ZIP_OUT)
except Exception as e:
    print("download manually from", ZIP_OUT, "|", e)

/content/EN_subtask1_submission.zip: 5,367 bytes | contains ['subtask_1_best_prediction.txt'] | 558 lines | OK


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# === AGGRESSIVE GPU CLEANUP: run, check the printout, then re-run the final EN cell ===
import gc, sys, warnings, torch
for _a in ("last_type", "last_value", "last_traceback", "last_exc"):
    if hasattr(sys, _a):
        setattr(sys, _a, None)
for _n in ["r", "_r", "res", "tr_live", "_tr", "tr_save", "tr_loaded", "_with", "_without", "_stopped"]:
    globals().pop(_n, None)
try:
    get_ipython().run_line_magic("reset", "-f out")
except Exception:
    pass
n_mod = n_opt = 0
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    _objs = gc.get_objects()
    for _o in _objs:
        try:
            if isinstance(_o, torch.optim.Optimizer):
                _o.state.clear(); n_opt += 1                       # Adam moments: 2x model size on the GPU
            elif isinstance(_o, torch.nn.Module):
                if any(p.is_cuda for p in _o.parameters(recurse=False)) or \
                   any(b.is_cuda for b in _o.buffers(recurse=False)):
                    _o.cpu(); n_mod += 1
        except Exception:
            continue
    del _objs
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        left = [t for t in gc.get_objects() if isinstance(t, torch.Tensor) and t.is_cuda]
        print(f"moved {n_mod} module(s) to CPU, cleared {n_opt} optimizer(s) | "
              f"GPU: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated, {torch.cuda.memory_reserved() / 1e9:.2f} GB reserved "
              f"| {len(left)} loose CUDA tensors ({sum(t.numel() * t.element_size() for t in left) / 1e9:.2f} GB)")
        del left

Flushing output cache (0 entries)


ReferenceError: weakly-referenced object no longer exists

In [ ]:
# === FINAL EN MODEL: E2 recipe, K-fold over EN dev (honest EN F1) + all other labelled data -> CodaBench zip ===
import gc, sys, zipfile, shutil, json
N_SEEDS_FINAL = 3          # number of folds = models (~20-30 min each on a T4); 2 is fine if time is short
BACKBONE_FINAL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
OUT_DIR = "/content/drive/MyDrive/mcpt_models/EN_fullfit_ensemble" if os.path.isdir("/content/drive/MyDrive") \
    else os.path.join(CFG.out_dir, "EN_fullfit_ensemble")
os.makedirs(OUT_DIR, exist_ok=True)

# ---- 1) switch OFF the best-model keeper ----
if "_orig_maybe_eval" in globals():
    MCPTTrainer._maybe_eval = _orig_maybe_eval
if "_orig_run_few_shot" in globals():
    run_few_shot = _orig_run_few_shot
CFG.eval_every = 0
CFG.p2_post_epochs = min(CFG.p2_post_epochs, 20)

# ---- 2) free GPU memory left by stopped cells ----
for _a in ("last_type", "last_value", "last_traceback", "last_exc"):
    if hasattr(sys, _a):
        setattr(sys, _a, None)
for _n in ["r", "_r", "tr_live", "_tr", "tr_save", "tr_loaded"]:
    globals().pop(_n, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ---- 3) folds over EN dev articles; all other languages train on train + their gold dev ----
from sklearn.model_selection import GroupKFold
N_FOLDS = N_SEEDS_FINAL
langs27 = ALL27 if "ALL27" in globals() else LANGS_2027
en_dev_all = SPLITS["EN"]["dev"].reset_index(drop=True)
Y_dev = np.stack(en_dev_all["labels"].values)
groups = en_dev_all["article_id"].astype(str).values
folds = list(GroupKFold(n_splits=N_FOLDS).split(en_dev_all, groups=groups))
BASE = {}
for l, d in SPLITS.items():
    BASE[l] = dict(d)
    if l != "EN" and len(d["dev"]):
        BASE[l]["train"] = _cat([d["train"], d["dev"]])

# ---- 4) one model per fold: EN train + the other EN dev folds; score the held-out fold ----
use_backbone(BACKBONE_FINAL)
en_src = [s for s in SOURCES["predict"] if s["lang"] == "EN"][0]
en_df = load_span_file(en_src, labelled=False)
P_oof = np.zeros((len(en_dev_all), NUM_CLASSES))
P_all, fold_rows = [], {}
for k, (tr_idx, ho_idx) in enumerate(folds):
    cfg_k = dataclasses.replace(CFG, seed=CFG.seed + k)
    FK = {l: dict(d) for l, d in BASE.items()}
    FK["EN"]["train"] = _cat([SPLITS["EN"]["train"], en_dev_all.iloc[tr_idx]])
    FK["EN"]["dev"] = en_dev_all.iloc[ho_idx].reset_index(drop=True)      # never trained on -> honest
    res = run_few_shot(cfg_k, FK, "EN", os.path.join(CFG.out_dir, "en_fullfit"), None,
                       train_langs=langs27, select=True, name=f"FULLFIT_fold{k}", save=False)
    P_oof[ho_idx] = res["trainer"].predict_proba(FK["EN"]["dev"])
    P = res["trainer"].predict_proba(en_df)
    np.save(os.path.join(OUT_DIR, f"probs_fold{k}.npy"), P)
    P_all.append(P)
    fold_rows[f"fold {k} ({len(ho_idx)} held-out paragraphs)"] = res["dev"]
    if k == 0:
        res["trainer"].enc.model.save_pretrained(os.path.join(OUT_DIR, "encoder_fold0"))
        TOKENIZER.save_pretrained(os.path.join(OUT_DIR, "encoder_fold0"))
        torch.save(res["trainer"].head.state_dict(), os.path.join(OUT_DIR, "head_fold0.pt"))
    del res["trainer"], res
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"fold {k + 1}/{N_FOLDS} done")
use_backbone(CFG.model_name)

# ---- 5) honest EN F1 from out-of-fold predictions + threshold for the submission ----
FINAL_THR, oof = tune_threshold(Y_dev, P_oof, CFG)
oof.update(crossfit_scores(Y_dev, P_oof, groups, CFG))
print("\nper fold (each scored on EN dev paragraphs it never saw):")
print(pd.DataFrame(fold_rows).T[["micro_f1", "macro_f1", "threshold"]].round(3).to_string())
print(f"\n===== EN F1 (honest, out-of-fold over all {len(en_dev_all)} EN dev paragraphs) =====")
print(f"  EN cf micro-F1 = {oof.get('cf_micro_f1', oof['micro_f1']):.4f}   <- the number to quote")
print(f"  EN micro-F1    = {oof['micro_f1']:.4f} (threshold {FINAL_THR} tuned on these predictions)")
print(f"  EN macro-F1    = {oof['macro_f1']:.4f} | cf macro-F1 {oof.get('cf_macro_f1', float('nan')):.4f}")
print(f"  for comparison: submitted model = 0.4527 on dev (honest) -> 0.4151 on the leaderboard")

P_mean = np.mean(P_all, 0)
Yh = binarize(P_mean, FINAL_THR, CFG.force_at_least_one)
pred_txt = os.path.join(OUT_DIR, "subtask_1_best_prediction.txt")
write_submission(en_df, Yh, pred_txt, CFG)
n_empty = int((Yh.sum(1) == 0).sum())
print(f"\nsubmission: {len(en_df)} paragraphs | {n_empty} with no framing ({n_empty / len(en_df):.0%}) | "
      f"mean labels per paragraph {Yh.sum(1).mean():.2f} (EN gold dev: 1.16, 29.5% empty)")
if not (0.10 <= n_empty / len(en_df) <= 0.50) or not (0.7 <= Yh.sum(1).mean() <= 2.0):
    print("!! WARNING: prediction profile is far from EN dev. Check before uploading.")
zip_path = "/content/EN_subtask1_fullfit_ensemble.zip" if os.path.isdir("/content") else os.path.join(OUT_DIR, "sub.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(pred_txt, arcname="subtask_1_best_prediction.txt")
with zipfile.ZipFile(zip_path) as z:
    assert z.testzip() is None and z.namelist() == ["subtask_1_best_prediction.txt"]
if os.path.dirname(os.path.abspath(zip_path)) != os.path.abspath(OUT_DIR):
    shutil.copy(zip_path, OUT_DIR)
json.dump({"backbone": BACKBONE_FINAL, "folds": N_FOLDS, "threshold": FINAL_THR, "en_dev_oof": oof,
           "training": "train + gold dev (EN dev via folds), BG/EN/PT/RU 2027", "recipe": "E2 (multi + epoch selection)"},
          open(os.path.join(OUT_DIR, "meta.json"), "w"), indent=1)
print("zip ready:", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("download manually from", zip_path, "|", e)

{'stage': 'FULLFIT_fold0_head_pretrain', 'epoch': 0, 'head_lr': 0.00099, 'enc_lr': 2e-05, 'bce': 0.3267}
{'stage': 'FULLFIT_fold0_head_pretrain', 'epoch': 1, 'head_lr': 0.0009801, 'enc_lr': 2e-05, 'bce': 0.2447}
{'stage': 'FULLFIT_fold0_head_pretrain', 'epoch': 2, 'head_lr': 0.0009703, 'enc_lr': 2e-05, 'bce': 0.2256}
{'stage': 'FULLFIT_fold0_head_pretrain', 'epoch': 3, 'head_lr': 0.0009606, 'enc_lr': 2e-05, 'bce': 0.2155}
{'stage': 'FULLFIT_fold0_head_pretrain', 'epoch': 4, 'head_lr': 0.000951, 'enc_lr': 2e-05, 'bce': 0.2091}
{'stage': 'FULLFIT_fold0_head_pretrain', 'epoch': 5, 'head_lr': 0.0009415, 'enc_lr': 2e-05, 'bce': 0.2023}
{'stage': 'FULLFIT_fold0_head_pretrain', 'epoch': 6, 'head_lr': 0.0009321, 'enc_lr': 2e-05, 'bce': 0.1967}
{'stage': 'FULLFIT_fold0_head_pretrain', 'epoch': 7, 'head_lr': 0.0009227, 'enc_lr': 2e-05, 'bce': 0.1927}
{'stage': 'FULLFIT_fold0_head_pretrain', 'epoch': 8, 'head_lr': 0.0009135, 'enc_lr': 2e-05, 'bce': 0.1875}
  >> FULLFIT_fold0_head_pretrain epoch 9

KeyboardInterrupt: 

In [ ]:
# === ZIP + DOWNLOAD THE BEST EN MODEL ===
import os, shutil, json
MODEL_DIR = "/content/drive/MyDrive/mcpt_models/EN_best_on_dev"
ZIP_BASE = "/content/EN_best_model"                      # -> /content/EN_best_model.zip

if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")
assert os.path.isdir(MODEL_DIR), f"not found: {MODEL_DIR}"

meta = json.load(open(os.path.join(MODEL_DIR, "meta.json")))
print(f"model: {meta.get('base_model')} | selected at {meta.get('selected_at')} | "
      f"EN cf micro-F1 {meta.get('en_cf_micro_f1', float('nan')):.4f} | threshold {meta.get('threshold')}")
for root, _, files_ in os.walk(MODEL_DIR):
    for f in files_:
        p = os.path.join(root, f)
        print(f"  {os.path.relpath(p, MODEL_DIR):60s} {os.path.getsize(p) / 1e6:9.1f} MB")

zip_path = shutil.make_archive(ZIP_BASE, "zip", root_dir=MODEL_DIR)   # folder contents at the zip root
print(f"\n{zip_path}: {os.path.getsize(zip_path) / 1e9:.2f} GB")
shutil.copy(zip_path, os.path.dirname(MODEL_DIR))                     # also keep the zip in Drive
print("copy in Drive:", os.path.join(os.path.dirname(MODEL_DIR), os.path.basename(zip_path)))

from google.colab import files
files.download(zip_path)s

model: sentence-transformers/paraphrase-multilingual-mpnet-base-v2 | selected at E2_mpnet_multi_sel_head_posttrain epoch 6 | EN cf micro-F1 0.4611 | threshold 0.3
  head.pt                                                            0.8 MB
  meta.json                                                          0.0 MB
  3_dev-set-nov-release__EN__subtask-1-predictions.txt               0.0 MB
  EN_subtask1_submission.zip                                         0.0 MB
  encoder/config.json                                                0.0 MB
  encoder/model.safetensors                                       1112.2 MB
  encoder/tokenizer_config.json                                      0.0 MB
  encoder/tokenizer.json                                            17.1 MB

/content/EN_best_model.zip: 1.01 GB
copy in Drive: /content/drive/MyDrive/mcpt_models/EN_best_model.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>